# Multi-Document Research Agent

A multi-document RAG research agent: hybrid (vector + BM25) search, conversational
memory, source citations, streaming Gemini generation, a REST API, a Streamlit chat
UI, an evaluation harness, Docker packaging, and CI. Everything below is generated
from this one Colab notebook, top to bottom.

What running it end-to-end does:

1. Installs the required libraries.
2. Writes the project to disk (every file in the README's folder structure) via
   `os.makedirs()` + `open(..., "w")`.
3. Builds a small example document set and a persisted FAISS vector database.
4. Runs the CLI, the FastAPI REST service, and the Streamlit UI as demos.
5. Runs the unit tests and a retrieval evaluation.
6. Zips the finished project so you can download it or push it to GitHub.

Note on the free tier: this notebook is built to work with a free Gemini API key.
Cell 3 sets `min_request_interval_s` (default 4s) so calls don't slam into the
per-minute rate limit, and query rewriting is off by default since it doubles the
number of Gemini calls per turn. If you're on a paid key, you can turn both back up
in `configs/config.yaml` or via env vars.


## 1. Project Setup — Install Libraries

In [ ]:

!pip install -q \
    "google-genai>=1.0.0" \
    "sentence-transformers>=3.0.0" \
    "faiss-cpu>=1.8.0" \
    "rank-bm25>=0.2.2" \
    "pypdf>=4.2.0" \
    "python-docx>=1.1.0" \
    "unstructured>=0.14.0" \
    "fastapi>=0.111.0" \
    "uvicorn[standard]>=0.30.0" \
    "streamlit>=1.35.0" \
    "python-multipart>=0.0.9" \
    "pydantic>=2.7.0" \
    "python-dotenv>=1.0.1" \
    "pyyaml>=6.0.1" \
    "numpy>=1.26.0" \
    "pandas>=2.2.0" \
    "scikit-learn>=1.4.0" \
    "tiktoken>=0.7.0" \
    "tqdm>=4.66.0" \
    "rich>=13.7.0" \
    "matplotlib>=3.8.0" \
    "plotly>=5.22.0" \
    "requests>=2.32.0" \
    "pytest>=8.2.0" \
    "httpx>=0.27.0" \
    nest_asyncio pyngrok

print("All libraries installed.")

## 2. Imports

In [ ]:
import os
import sys
import json
import shutil
import subprocess
import time
from pathlib import Path

print("Base imports ready.")

## 3. Configuration

Paste your Gemini API key below. Get a free key at
https://aistudio.google.com/app/apikey — the rest of the notebook works
end-to-end once this is set. (Ingestion, retrieval, and unit-test cells also
work *without* a key, since they don't call the Gemini API.)

In [ ]:
import os
from getpass import getpass

# Paste your key when prompted (input is hidden) instead of hardcoding it here,
# so it never ends up saved in plain text inside this .ipynb file.
if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Gemini API key: ")


In [ ]:
import os
from pathlib import Path

# =============================================================================
# Project Configuration
# =============================================================================

PROJECT_NAME = "03_Multi_Document_Research_Agent"
PROJECT_ROOT = Path.cwd() / PROJECT_NAME

GEMINI_MODEL = "gemini-3.5-flash"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# =============================================================================
# API Key (Never hardcode secrets)
# =============================================================================

# Set this in Colab before running:
# %env GEMINI_API_KEY=your_api_key_here
# OR
# os.environ["GEMINI_API_KEY"] = "your_api_key"

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY or GEMINI_API_KEY == "PASTE_YOUR_API_KEY_HERE":
    raise ValueError(
        " GEMINI_API_KEY not found.\n"
        "Get a free key at https://aistudio.google.com/app/apikey, then run this "
        "once before executing the rest of the notebook:\n\n"
        '%env GEMINI_API_KEY=YOUR_API_KEY'
    )

# Export project-wide environment variables
os.environ["GEMINI_MODEL"] = GEMINI_MODEL
os.environ["EMBEDDING_MODEL"] = EMBEDDING_MODEL

print("Gemini API key loaded securely.")
print(f"Project will be generated at: {PROJECT_ROOT}")

## 4. Create Folder Structure

In [ ]:
folders = [
    "",
    "configs",
    "data/documents",
    "data/vectorstore",
    "logs",
    "tests",
    "assets",
    "notebooks",
    "src",
    ".github/workflows",
]

for folder in folders:
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)

# .gitkeep placeholders so empty runtime dirs are still tracked by git
for keep_dir in ["data/documents", "data/vectorstore", "logs", "assets", "notebooks"]:
    (PROJECT_ROOT / keep_dir / ".gitkeep").touch()

print("Folder structure created:")
for folder in folders:
    print("  -", PROJECT_ROOT / folder)

## 5. Generate Files — Root Project Files

Each cell below writes one complete file to disk. This is the pattern used
throughout the notebook: `os.makedirs(...)` + `open(...,"w").write(...)`.

In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent", exist_ok=True)

_content = '#  Multi-Document Research Agent\n\n[![CI](https://img.shields.io/badge/CI-GitHub_Actions-blue?logo=githubactions&logoColor=white)](.github/workflows/python.yml)\n[![Python](https://img.shields.io/badge/Python-3.11-blue?logo=python&logoColor=white)](https://www.python.org/)\n[![License: MIT](https://img.shields.io/badge/License-MIT-green.svg)](LICENSE)\n[![Gemini](https://img.shields.io/badge/LLM-Gemini_3.5_Flash-orange)](https://ai.google.dev/)\n[![FAISS](https://img.shields.io/badge/Vector_DB-FAISS-purple)](https://github.com/facebookresearch/faiss)\n[![FastAPI](https://img.shields.io/badge/API-FastAPI-009688?logo=fastapi&logoColor=white)](https://fastapi.tiangolo.com/)\n[![Streamlit](https://img.shields.io/badge/UI-Streamlit-FF4B4B?logo=streamlit&logoColor=white)](https://streamlit.io/)\n\nAn enterprise-grade **Retrieval-Augmented Generation (RAG)** system for querying across\nmultiple PDF, DOCX, and TXT documents at once — with hybrid (vector + keyword) search,\nconversational memory, source citations, and a Gemini-powered answer engine.\n\nBuilt to go beyond a "toy PDF chatbot": modular architecture, persistence, incremental\nindexing, evaluation harness, REST API, chat UI, Docker packaging, and CI.\n\n---\n\n##  Features\n\n| Category | Capabilities |\n|---|---|\n| **Ingestion** | Multi-file PDF / DOCX / TXT upload, automatic loading, text cleaning, metadata extraction |\n| **Chunking** | Recursive & semantic (sentence-aware) strategies, configurable size/overlap |\n| **Retrieval** | FAISS vector search + BM25 keyword search fused via Reciprocal Rank Fusion, metadata filtering, near-duplicate removal, context compression |\n| **Generation** | Gemini `3.5-flash`, streaming responses, query rewriting for follow-ups, conversation memory, inline citations |\n| **Interfaces** | CLI (`main.py`), REST API (`api.py`, FastAPI + Swagger), Chat UI (`streamlit_app.py`) |\n| **Ops** | Structured logging, Docker + docker-compose, GitHub Actions CI, persistence & incremental indexing |\n| **Evaluation** | Precision@K, Recall@K, MRR, latency benchmarking, embedding-space & timing visualizations |\n| **Quality** | Type hints, docstrings throughout, unit tests (pytest), PEP8-formatted |\n\n---\n\n##  Architecture\n\n```\n                         ┌──────────────────────────┐\n                         │   Documents (PDF/DOCX/TXT) │\n                         └─────────────┬────────────┘\n                                        ▼\n                         ┌──────────────────────────┐\n                         │   document_loader.py      │  clean + extract metadata\n                         └─────────────┬────────────┘\n                                        ▼\n                         ┌──────────────────────────┐\n                         │   chunker.py              │  recursive / semantic split\n                         └─────────────┬────────────┘\n                                        ▼\n                         ┌──────────────────────────┐\n                         │   embedder.py             │  sentence-transformers\n                         └─────────────┬────────────┘\n                                        ▼\n                         ┌──────────────────────────┐\n                         │   vector_store.py (FAISS) │  persisted, incremental\n                         └─────────────┬────────────┘\n                                        │\n      ┌─────────────────────────────────┼─────────────────────────────────┐\n      ▼                                                                   ▼\n┌───────────────┐                                              ┌───────────────────┐\n│ reranker.py    │  BM25 index + Reciprocal Rank Fusion         │ retriever.py       │\n│ (lexical side) │◄─────────────────────────────────────────────┤ (hybrid orchestr.) │\n└───────────────┘                                              └─────────┬─────────┘\n                                                                          ▼\n                                        ┌────────────────────────────────────────────┐\n                                        │  dedup → metadata filter → context compress │\n                                        └───────────────────────┬────────────────────┘\n                                                                 ▼\n                                        ┌────────────────────────────────────────────┐\n                                        │  prompts.py + memory.py → gemini_client.py  │\n                                        └───────────────────────┬────────────────────┘\n                                                                 ▼\n                                                   ┌─────────────────────────┐\n                                                   │  Answer + Citations      │\n                                                   └─────────────────────────┘\n\n                     Exposed via:  main.py (CLI)  ·  api.py (FastAPI)  ·  streamlit_app.py (UI)\n```\n\n---\n\n##  Screenshots\n\n> _Add screenshots after running the app locally:_\n> `assets/screenshot_streamlit_chat.png` · `assets/screenshot_swagger_ui.png` · `assets/screenshot_cli.png`\n\n---\n\n##  Google Colab Instructions\n\n1. Open the notebook `03_Multi_Document_Research_Agent.ipynb` in Google Colab.\n2. Run every cell top-to-bottom — this **generates the entire project** on disk\n   (`os.makedirs` + file writes), installs dependencies, builds a sample vector\n   database, and demonstrates the CLI, API, and evaluation flows.\n3. Paste your Gemini API key into the **Configuration** cell:\n   ```python\n   GEMINI_API_KEY = "PASTE_YOUR_API_KEY_HERE"\n   ```\n   Get a free key at https://aistudio.google.com/app/apikey.\n4. The final cell zips the generated project so you can download it and push\n   it straight to GitHub.\n\n---\n\n##  Local Setup\n\n```bash\ngit clone https://github.com/<your-username>/03_Multi_Document_Research_Agent.git\ncd 03_Multi_Document_Research_Agent\n\npython3 -m venv venv\nsource venv/bin/activate        # Windows: venv\\Scripts\\activate\n\npip install -r requirements.txt\ncp .env.example .env            # then edit .env and paste your GEMINI_API_KEY\n```\n\n### Docker\n\n```bash\ncp .env.example .env            # edit with your API key\ndocker compose up --build       # API on :8000, Streamlit UI on :8501\n```\n\n---\n\n## \u200d Usage\n\n### CLI\n\n```bash\n# Ingest every document in data/documents/\npython main.py ingest --path data/documents\n\n# Ask a single question\npython main.py query "What is the refund policy?"\n\n# Interactive chat session (with memory)\npython main.py chat\n\n# Run retrieval evaluation\npython main.py evaluate\n```\n\n### REST API\n\n```bash\nuvicorn api:app --reload --port 8000\n# Swagger UI:  http://localhost:8000/docs\n```\n\n| Method | Endpoint | Description |\n|---|---|---|\n| `GET`    | `/health`             | Liveness check |\n| `GET`    | `/config`              | Current (redacted) configuration |\n| `POST`   | `/upload`               | Upload & index one or more documents |\n| `POST`   | `/query`                | Ask a question, get an answer + citations |\n| `DELETE` | `/documents/{doc_id}`   | Remove a document from the index |\n| `POST`   | `/reindex`               | Rebuild the index from `data/documents/` |\n\nExample query:\n```bash\ncurl -X POST http://localhost:8000/query \\\n  -H "Content-Type: application/json" \\\n  -d \'{"question": "What is retrieval augmented generation?", "top_k": 5}\'\n```\n\n### Streamlit Chat UI\n\n```bash\nstreamlit run streamlit_app.py\n```\nUpload documents from the sidebar, adjust the vector/BM25 balance and Top-K live,\nand chat with full source citations and response timing per turn.\n\n---\n\n##  Folder Structure\n\n```\n03_Multi_Document_Research_Agent/\n├── README.md\n├── LICENSE\n├── .gitignore\n├── requirements.txt\n├── Dockerfile\n├── docker-compose.yml\n├── .env.example\n├── setup.py\n├── main.py                 # CLI entry point\n├── api.py                  # FastAPI REST service\n├── streamlit_app.py        # Streamlit chat UI\n├── configs/\n│   └── config.yaml\n├── data/\n│   ├── documents/           # Source documents (gitignored contents)\n│   └── vectorstore/         # Persisted FAISS index (gitignored contents)\n├── logs/                    # Timestamped log files\n├── tests/                   # pytest unit tests\n├── assets/                  # Screenshots / generated charts\n├── notebooks/                # This Colab notebook\n├── src/\n│   ├── config.py             # Configuration system\n│   ├── logger.py              # Centralised logging\n│   ├── utils.py                # Shared helpers\n│   ├── document_loader.py       # PDF / DOCX / TXT loading + cleaning\n│   ├── chunker.py                # Recursive & semantic chunking\n│   ├── embedder.py                # SentenceTransformers wrapper\n│   ├── vector_store.py             # FAISS persistence & incremental indexing\n│   ├── reranker.py                  # BM25 + Reciprocal Rank Fusion\n│   ├── retriever.py                  # Hybrid retrieval orchestration\n│   ├── memory.py                      # Conversation memory\n│   ├── prompts.py                      # Prompt templates\n│   ├── gemini_client.py                 # Gemini API wrapper (retry + streaming)\n│   ├── pipeline.py                       # End-to-end RAG orchestrator\n│   ├── evaluation.py                      # Precision@K / Recall@K / latency\n│   └── visualization.py                    # Plotly charts\n└── .github/workflows/python.yml              # CI: lint + test\n```\n\n---\n\n##  Technologies\n\nPython 3.11 · Google Gemini API (`gemini-3.5-flash`) · FAISS · sentence-transformers ·\nrank-bm25 · FastAPI · Streamlit · Pydantic · pandas / numpy / scikit-learn · Plotly ·\npypdf · python-docx · Docker\n\n---\n\n##  Performance\n\nMeasured on the bundled example dataset (small corpus, CPU-only, `all-MiniLM-L6-v2`):\n\n| Metric | Typical value |\n|---|---|\n| Embedding (per chunk) | ~5–15 ms |\n| Vector search (FAISS, flat index) | < 5 ms |\n| BM25 search | < 5 ms |\n| End-to-end query (retrieval + generation) | 1–3 s (dominated by the Gemini API call) |\n\nRun `python main.py evaluate` or the notebook\'s **Evaluation** section to benchmark\nagainst your own labelled query set and generate latency/embedding visualizations.\n\n---\n\n##  Future Improvements\n\n- Swap `IndexFlatIP` for an approximate index (`IndexIVFFlat` / `IndexHNSWFlat`) for\nmillion-scale corpora.\n- Add a real cross-encoder reranker (e.g. `ms-marco-MiniLM`) as a drop-in alternative\nto the current RRF-based fusion.\n- Multi-tenant document namespaces / access control in the API.\n- Async ingestion queue for very large document batches.\n- Structured output (JSON mode) for downstream tool integrations.\n\n---\n\n##  License\n\nReleased under the [MIT License](LICENSE).\n\n##  Author\n\nBuilt as a portfolio-ready reference implementation of a production RAG system.\nContributions and issues welcome.\n'

with open(r"03_Multi_Document_Research_Agent/README.md", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/README.md")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent", exist_ok=True)

_content = 'MIT License\n\nCopyright (c) 2026 Multi-Document Research Agent Contributors\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n'

with open(r"03_Multi_Document_Research_Agent/LICENSE", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/LICENSE")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent", exist_ok=True)

_content = "# Environments\n.env\nvenv/\nenv/\n__pycache__/\n*.pyc\n\n# Vector store & data (regenerate via ingestion, don't commit)\ndata/vectorstore/*\n!data/vectorstore/.gitkeep\ndata/documents/*\n!data/documents/.gitkeep\n\n# Logs\nlogs/*\n!logs/.gitkeep\n\n# IDE / OS\n.vscode/\n.idea/\n.DS_Store\n\n# Testing\n.pytest_cache/\n.coverage\nhtmlcov/\n\n# Streamlit\n.streamlit/secrets.toml\n\n# Packaging\n*.egg-info/\nbuild/\ndist/\n"

with open(r"03_Multi_Document_Research_Agent/.gitignore", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/.gitignore")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent", exist_ok=True)

_content = '# --- LLM / Embeddings -------------------------------------------------\ngoogle-genai>=1.0.0\nsentence-transformers>=3.0.0\n\n# --- Retrieval ----------------------------------------------------------\nfaiss-cpu>=1.8.0\nrank-bm25>=0.2.2\n\n# --- Document parsing -----------------------------------------------------\npypdf>=4.2.0\npython-docx>=1.1.0\nunstructured>=0.14.0\n\n# --- Web frameworks ---------------------------------------------------\nfastapi>=0.111.0\nuvicorn[standard]>=0.30.0\nstreamlit>=1.35.0\npython-multipart>=0.0.9\n\n# --- Data / config / utils --------------------------------------------\npydantic>=2.7.0\npython-dotenv>=1.0.1\npyyaml>=6.0.1\nnumpy>=1.26.0\npandas>=2.2.0\nscikit-learn>=1.4.0\ntiktoken>=0.7.0\ntqdm>=4.66.0\nrich>=13.7.0\n\n# --- Visualization ------------------------------------------------------\nmatplotlib>=3.8.0\nplotly>=5.22.0\n\n# --- HTTP -----------------------------------------------------------------\nrequests>=2.32.0\n\n# --- Testing --------------------------------------------------------------\npytest>=8.2.0\nhttpx>=0.27.0\n'

with open(r"03_Multi_Document_Research_Agent/requirements.txt", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/requirements.txt")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent", exist_ok=True)

_content = '# Copy this file to .env and fill in your own values.\n# NEVER commit your real .env file to version control.\n\n# Required: your Gemini API key (https://aistudio.google.com/app/apikey)\nGEMINI_API_KEY=PASTE_YOUR_API_KEY_HERE\n\n# Model configuration\nGEMINI_MODEL=gemini-3.5-flash\nEMBEDDING_MODEL=sentence-transformers/all-MiniLM-L6-v2\n\n# Chunking\nCHUNK_SIZE=800\nCHUNK_OVERLAP=120\nCHUNKING_STRATEGY=recursive\n\n# Retrieval\nTOP_K=5\nHYBRID_ALPHA=0.6\nMAX_CONTEXT_TOKENS=3000\nTEMPERATURE=0.3\n\n# Paths (usually fine to leave as defaults)\nVECTORSTORE_DIR=data/vectorstore\nDOCUMENTS_DIR=data/documents\nLOGS_DIR=logs\n'

with open(r"03_Multi_Document_Research_Agent/.env.example", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/.env.example")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent", exist_ok=True)

_content = '"""Packaging metadata for `pip install -e .` (editable/local install)."""\n\nfrom setuptools import find_packages, setup\n\nwith open("requirements.txt", encoding="utf-8") as fh:\n    requirements = [line.strip() for line in fh if line.strip() and not line.startswith("#")]\n\nsetup(\n    name="multi-document-research-agent",\n    version="1.0.0",\n    description="Enterprise-grade multi-document RAG agent with hybrid search, powered by Gemini.",\n    author="Your Name",\n    license="MIT",\n    packages=find_packages(include=["src", "src.*"]),\n    install_requires=requirements,\n    python_requires=">=3.10",\n    entry_points={\n        "console_scripts": [\n            "research-agent=main:main",\n        ],\n    },\n)\n'

with open(r"03_Multi_Document_Research_Agent/setup.py", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/setup.py")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent", exist_ok=True)

_content = '# syntax=docker/dockerfile:1\nFROM python:3.11-slim\n\nWORKDIR /app\n\n# System deps needed by faiss / sentence-transformers / pdf parsing\nRUN apt-get update && apt-get install -y --no-install-recommends \\\n    build-essential \\\n    curl \\\n    && rm -rf /var/lib/apt/lists/*\n\nCOPY requirements.txt .\nRUN pip install --no-cache-dir -r requirements.txt\n\nCOPY . .\n\nRUN mkdir -p data/documents data/vectorstore logs\n\nEXPOSE 8000 8501\n\n# Default: run the FastAPI service. Override CMD to run Streamlit instead:\n#   docker run <image> streamlit run streamlit_app.py --server.address=0.0.0.0\nCMD ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"]\n'

with open(r"03_Multi_Document_Research_Agent/Dockerfile", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/Dockerfile")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent", exist_ok=True)

_content = 'version: "3.9"\n\nservices:\napi:\n    build: .\n    image: multi-document-research-agent:latest\n    container_name: research-agent-api\n    ports:\n      - "8000:8000"\n    env_file:\n      - .env\n    volumes:\n      - ./data:/app/data\n      - ./logs:/app/logs\n    command: ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"]\n    restart: unless-stopped\n\nstreamlit:\n    build: .\n    image: multi-document-research-agent:latest\n    container_name: research-agent-ui\n    ports:\n      - "8501:8501"\n    env_file:\n      - .env\n    volumes:\n      - ./data:/app/data\n      - ./logs:/app/logs\n    command: ["streamlit", "run", "streamlit_app.py", "--server.address=0.0.0.0", "--server.port=8501"]\n    depends_on:\n      - api\n    restart: unless-stopped\n'

with open(r"03_Multi_Document_Research_Agent/docker-compose.yml", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/docker-compose.yml")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent/.github/workflows", exist_ok=True)

_content = 'name: CI\n\non:\npush:\n    branches: [main]\npull_request:\n    branches: [main]\n\njobs:\ntest:\n    runs-on: ubuntu-latest\n    strategy:\n      matrix:\n        python-version: ["3.10", "3.11"]\n\n    steps:\n      - uses: actions/checkout@v4\n\n      - name: Set up Python ${{ matrix.python-version }}\n        uses: actions/setup-python@v5\n        with:\n          python-version: ${{ matrix.python-version }}\n          cache: "pip"\n\n      - name: Install dependencies\n        run: |\n          python -m pip install --upgrade pip\n          pip install -r requirements.txt\n          pip install ruff\n\n      - name: Lint with ruff\n        run: ruff check src/ main.py api.py streamlit_app.py --output-format=github\n\n      - name: Run unit tests\n        run: pytest tests/ -v\n'

with open(r"03_Multi_Document_Research_Agent/.github/workflows/python.yml", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/.github/workflows/python.yml")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent/configs", exist_ok=True)

_content = '# Default application configuration.\n# Values here are overridden by environment variables (see .env.example),\n# which are in turn overridden by explicit keyword args to load_config().\n\ngemini_model: gemini-3.5-flash\nembedding_model: sentence-transformers/all-MiniLM-L6-v2\n\nchunk_size: 800\nchunk_overlap: 120\nchunking_strategy: recursive   # recursive | semantic\n\ntop_k: 5\nhybrid_alpha: 0.6              # 1.0 = pure vector search, 0.0 = pure BM25\nmax_context_tokens: 3000\ntemperature: 0.3\n\nrequest_timeout: 60\nmax_retries: 3\n'

with open(r"03_Multi_Document_Research_Agent/configs/config.yaml", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/configs/config.yaml")


## 6. Generate Source Code — the `src/` package

The core, modular RAG engine: configuration, logging, document loading,
chunking, embeddings, the FAISS vector store, hybrid (BM25 + vector) search
with reciprocal rank fusion, conversation memory, prompt templates, the
Gemini client, the end-to-end pipeline orchestrator, evaluation metrics, and
visualization helpers.

In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent/src", exist_ok=True)

_content = ''

with open(r"03_Multi_Document_Research_Agent/src/__init__.py", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/src/__init__.py")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent/src", exist_ok=True)

_content = '"""\nconfig.py\n---------\nCentral configuration system for the Multi-Document Research Agent.\n\nConfiguration is resolved in the following priority order (highest wins):\n    1. Explicit keyword arguments passed to `AppConfig(...)`\n    2. Environment variables (loaded from `.env` via python-dotenv)\n    3. `configs/config.yaml` file values\n    4. Hard-coded defaults defined below\n\nThe system is intentionally dependency-light: YAML is optional, and the\nwhole module degrades gracefully if `configs/config.yaml` is missing.\n"""\n\nfrom __future__ import annotations\n\nimport os\nfrom dataclasses import dataclass, field, asdict\nfrom pathlib import Path\nfrom typing import Any, Dict, Optional\n\ntry:\n    from dotenv import load_dotenv\n    load_dotenv()\nexcept ImportError:  # pragma: no cover - dotenv is a soft dependency\n    pass\n\ntry:\n    import yaml\nexcept ImportError:  # pragma: no cover - yaml is a soft dependency\n    yaml = None\n\n\nPROJECT_ROOT = Path(__file__).resolve().parent.parent\n\n\ndef _load_yaml_config(path: Path) -> Dict[str, Any]:\n    """Load a YAML config file if it exists, returning an empty dict otherwise."""\n    if yaml is None or not path.exists():\n        return {}\n    with open(path, "r", encoding="utf-8") as fh:\n        data = yaml.safe_load(fh) or {}\n    return data\n\n\n@dataclass\nclass AppConfig:\n    """Strongly-typed application configuration.\n\n    Attributes:\n        gemini_api_key: API key used to authenticate against the Gemini API.\n        gemini_model: Model name used for generation (e.g. "gemini-3.5-flash").\n        embedding_model: SentenceTransformers model used to embed text.\n        chunk_size: Target number of characters per chunk.\n        chunk_overlap: Number of overlapping characters between consecutive chunks.\n        chunking_strategy: Either "recursive" or "semantic".\n        top_k: Default number of chunks to retrieve per query.\n        hybrid_alpha: Weight given to vector search vs BM25 in [0, 1].\n                      1.0 = pure vector search, 0.0 = pure BM25.\n        max_context_tokens: Maximum number of tokens allowed in the assembled\n                             context that is sent to the LLM.\n        temperature: Sampling temperature for generation.\n        vectorstore_dir: Directory where the FAISS index + metadata are stored.\n        documents_dir: Directory where uploaded/source documents live.\n        logs_dir: Directory where log files are written.\n        request_timeout: Timeout (seconds) for outbound Gemini API calls.\n        max_retries: Number of retry attempts for transient API failures.\n    """\n\n    gemini_api_key: str = ""\n    gemini_model: str = "gemini-2.5-flash-lite"\n    embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2"\n\n    chunk_size: int = 800\n    chunk_overlap: int = 120\n    chunking_strategy: str = "recursive"  # "recursive" | "semantic"\n\n    top_k: int = 5\n    hybrid_alpha: float = 0.6\n    max_context_tokens: int = 3000\n    temperature: float = 0.3\n\n    vectorstore_dir: str = str(PROJECT_ROOT / "data" / "vectorstore")\n    documents_dir: str = str(PROJECT_ROOT / "data" / "documents")\n    logs_dir: str = str(PROJECT_ROOT / "logs")\n\n    request_timeout: int = 60\n    max_retries: int = 3\n\n    # Free-tier friendliness: a free Gemini key allows only a handful of\n    # requests per minute. min_request_interval_s spaces calls out so we do\n    # not trip a 429, and enable_query_rewrite is off by default since it\n    # costs a whole extra Gemini call per turn.\n    min_request_interval_s: float = 4.0\n    enable_query_rewrite: bool = False\n\n    def __post_init__(self) -> None:\n        Path(self.vectorstore_dir).mkdir(parents=True, exist_ok=True)\n        Path(self.documents_dir).mkdir(parents=True, exist_ok=True)\n        Path(self.logs_dir).mkdir(parents=True, exist_ok=True)\n\n    def to_dict(self) -> Dict[str, Any]:\n        """Return a JSON-serialisable dict, masking the API key."""\n        data = asdict(self)\n        if data.get("gemini_api_key"):\n            data["gemini_api_key"] = "***REDACTED***"\n        return data\n\n\ndef load_config(yaml_path: Optional[str] = None, **overrides: Any) -> AppConfig:\n    """Build an `AppConfig`, merging YAML file, environment variables, and overrides.\n\n    Args:\n        yaml_path: Optional explicit path to a YAML config file. Defaults to\n                   `configs/config.yaml` relative to the project root.\n        **overrides: Explicit keyword overrides, take highest priority.\n\n    Returns:\n        A fully resolved `AppConfig` instance.\n    """\n    yaml_file = Path(yaml_path) if yaml_path else PROJECT_ROOT / "configs" / "config.yaml"\n    yaml_values = _load_yaml_config(yaml_file)\n\n    env_values: Dict[str, Any] = {}\n    env_map = {\n        "GEMINI_API_KEY": "gemini_api_key",\n        "GEMINI_MODEL": "gemini_model",\n        "EMBEDDING_MODEL": "embedding_model",\n        "CHUNK_SIZE": ("chunk_size", int),\n        "CHUNK_OVERLAP": ("chunk_overlap", int),\n        "CHUNKING_STRATEGY": "chunking_strategy",\n        "TOP_K": ("top_k", int),\n        "HYBRID_ALPHA": ("hybrid_alpha", float),\n        "MAX_CONTEXT_TOKENS": ("max_context_tokens", int),\n        "MIN_REQUEST_INTERVAL_S": ("min_request_interval_s", float),\n        "ENABLE_QUERY_REWRITE": ("enable_query_rewrite", lambda v: v.lower() in ("1", "true", "yes")),\n        "TEMPERATURE": ("temperature", float),\n        "VECTORSTORE_DIR": "vectorstore_dir",\n        "DOCUMENTS_DIR": "documents_dir",\n        "LOGS_DIR": "logs_dir",\n    }\n    for env_key, target in env_map.items():\n        raw = os.getenv(env_key)\n        if raw is None:\n            continue\n        if isinstance(target, tuple):\n            field_name, caster = target\n            env_values[field_name] = caster(raw)\n        else:\n            env_values[target] = raw\n\n    merged = {**yaml_values, **env_values, **overrides}\n    valid_fields = {f for f in AppConfig.__dataclass_fields__}\n    merged = {k: v for k, v in merged.items() if k in valid_fields}\n    return AppConfig(**merged)\n\n\n# Module-level singleton, lazily created the first time it\'s imported by the app.\nconfig = load_config()\n'

with open(r"03_Multi_Document_Research_Agent/src/config.py", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/src/config.py")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent/src", exist_ok=True)

_content = '"""\nlogger.py\n---------\nCentralised logging setup. Every module in the project should call\n`get_logger(__name__)` rather than instantiating its own logger, so that\nformatting, log level, and log file destination stay consistent.\n"""\n\nfrom __future__ import annotations\n\nimport logging\nfrom datetime import datetime\nfrom pathlib import Path\nfrom typing import Optional\n\ntry:\n    from rich.logging import RichHandler\n    _HAS_RICH = True\nexcept ImportError:  # pragma: no cover\n    _HAS_RICH = False\n\nfrom src.config import config\n\n_CONFIGURED_LOGGERS = set()\n\n\ndef get_logger(name: str, log_file: Optional[str] = None) -> logging.Logger:\n    """Return a configured logger that writes to console (rich, if available)\n    and to a timestamped file under `logs/`.\n\n    Args:\n        name: Usually `__name__` of the calling module.\n        log_file: Optional override for the log filename.\n\n    Returns:\n        A ready-to-use `logging.Logger`.\n    """\n    logger = logging.getLogger(name)\n\n    if name in _CONFIGURED_LOGGERS:\n        return logger\n\n    logger.setLevel(logging.INFO)\n    logger.propagate = False\n\n    formatter = logging.Formatter(\n        fmt="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",\n        datefmt="%Y-%m-%d %H:%M:%S",\n    )\n\n    if _HAS_RICH:\n        console_handler: logging.Handler = RichHandler(rich_tracebacks=True, show_path=False)\n    else:  # pragma: no cover\n        console_handler = logging.StreamHandler()\n        console_handler.setFormatter(formatter)\n    logger.addHandler(console_handler)\n\n    logs_dir = Path(config.logs_dir)\n    logs_dir.mkdir(parents=True, exist_ok=True)\n    file_name = log_file or f"app_{datetime.now().strftime(\'%Y%m%d\')}.log"\n    file_handler = logging.FileHandler(logs_dir / file_name, encoding="utf-8")\n    file_handler.setFormatter(formatter)\n    logger.addHandler(file_handler)\n\n    _CONFIGURED_LOGGERS.add(name)\n    return logger\n'

with open(r"03_Multi_Document_Research_Agent/src/logger.py", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/src/logger.py")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent/src", exist_ok=True)

_content = '"""\nutils.py\n--------\nSmall, dependency-light helper utilities shared across the project:\ntext cleaning, hashing, timing, and filesystem helpers.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport re\nimport time\nfrom contextlib import contextmanager\nfrom pathlib import Path\nfrom typing import Iterator\n\n\ndef ensure_dir(path: str | Path) -> Path:\n    """Create a directory (and parents) if it doesn\'t already exist."""\n    p = Path(path)\n    p.mkdir(parents=True, exist_ok=True)\n    return p\n\n\ndef clean_text(text: str) -> str:\n    """Normalise whitespace and strip control characters from raw extracted text.\n\n    - Collapses runs of whitespace into single spaces (but preserves paragraph\n      breaks as double newlines).\n    - Removes null bytes and other non-printable control characters.\n    """\n    if not text:\n        return ""\n    text = text.replace("\\x00", "")\n    text = re.sub(r"[ \\t]+", " ", text)\n    text = re.sub(r"\\n{3,}", "\\n\\n", text)\n    text = "\\n".join(line.strip() for line in text.split("\\n"))\n    return text.strip()\n\n\ndef stable_hash(text: str, length: int = 12) -> str:\n    """Return a short, stable hex hash for a piece of text (used for IDs)."""\n    digest = hashlib.sha256(text.encode("utf-8")).hexdigest()\n    return digest[:length]\n\n\ndef generate_doc_id(source_path: str) -> str:\n    """Generate a stable document ID derived from its source path."""\n    return f"doc_{stable_hash(source_path)}"\n\n\ndef generate_chunk_id(doc_id: str, chunk_index: int) -> str:\n    """Generate a deterministic chunk ID from a document ID and chunk position."""\n    return f"{doc_id}_chunk_{chunk_index:04d}"\n\n\n@contextmanager\ndef timer() -> Iterator[dict]:\n    """Context manager that measures elapsed wall-clock time.\n\n    Usage:\n        with timer() as t:\n            do_work()\n        print(t["elapsed_seconds"])\n    """\n    state = {"elapsed_seconds": 0.0}\n    start = time.perf_counter()\n    try:\n        yield state\n    finally:\n        state["elapsed_seconds"] = time.perf_counter() - start\n\n\ndef truncate(text: str, max_chars: int = 200) -> str:\n    """Truncate text for display purposes, appending an ellipsis if needed."""\n    text = text.strip()\n    return text if len(text) <= max_chars else text[: max_chars - 1].rstrip() + "…"\n'

with open(r"03_Multi_Document_Research_Agent/src/utils.py", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/src/utils.py")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent/src", exist_ok=True)

_content = '"""\ndocument_loader.py\n-------------------\nLoads raw documents (PDF, DOCX, TXT) from disk, extracts text + metadata,\nand cleans the extracted text so it is ready for chunking.\n"""\n\nfrom __future__ import annotations\n\nimport os\nfrom dataclasses import dataclass, field\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any, Dict, List\n\nfrom src.logger import get_logger\nfrom src.utils import clean_text, generate_doc_id\n\nlogger = get_logger(__name__)\n\nSUPPORTED_EXTENSIONS = {".pdf", ".docx", ".txt", ".md"}\n\n\n@dataclass\nclass Document:\n    """A single loaded source document."""\n\n    doc_id: str\n    source: str\n    text: str\n    metadata: Dict[str, Any] = field(default_factory=dict)\n\n\nclass DocumentLoadError(Exception):\n    """Raised when a document cannot be parsed."""\n\n\ndef _load_txt(path: Path) -> str:\n    return path.read_text(encoding="utf-8", errors="ignore")\n\n\ndef _load_pdf(path: Path) -> str:\n    try:\n        from pypdf import PdfReader\n    except ImportError as exc:  # pragma: no cover\n        raise DocumentLoadError("pypdf is required to load PDF files") from exc\n\n    reader = PdfReader(str(path))\n    pages = []\n    for page in reader.pages:\n        pages.append(page.extract_text() or "")\n    return "\\n\\n".join(pages)\n\n\ndef _load_docx(path: Path) -> str:\n    try:\n        import docx  # python-docx\n    except ImportError as exc:  # pragma: no cover\n        raise DocumentLoadError("python-docx is required to load DOCX files") from exc\n\n    document = docx.Document(str(path))\n    paragraphs = [p.text for p in document.paragraphs if p.text.strip()]\n    return "\\n\\n".join(paragraphs)\n\n\n_LOADERS = {\n    ".txt": _load_txt,\n    ".md": _load_txt,\n    ".pdf": _load_pdf,\n    ".docx": _load_docx,\n}\n\n\ndef load_document(path: str | Path) -> Document:\n    """Load a single document from disk into a `Document` object.\n\n    Args:\n        path: Path to a .pdf, .docx, .txt, or .md file.\n\n    Returns:\n        A `Document` with cleaned text and extracted metadata.\n\n    Raises:\n        DocumentLoadError: If the file type is unsupported or parsing fails.\n    """\n    path = Path(path)\n    ext = path.suffix.lower()\n\n    if ext not in _LOADERS:\n        raise DocumentLoadError(\n            f"Unsupported file type \'{ext}\'. Supported: {sorted(SUPPORTED_EXTENSIONS)}"\n        )\n    if not path.exists():\n        raise DocumentLoadError(f"File not found: {path}")\n\n    try:\n        raw_text = _LOADERS[ext](path)\n    except DocumentLoadError:\n        raise\n    except Exception as exc:  # noqa: BLE001 - surface as a DocumentLoadError\n        raise DocumentLoadError(f"Failed to parse {path.name}: {exc}") from exc\n\n    text = clean_text(raw_text)\n    stat = path.stat()\n    metadata = {\n        "filename": path.name,\n        "file_type": ext.lstrip("."),\n        "file_size_bytes": stat.st_size,\n        "num_characters": len(text),\n        "num_words": len(text.split()),\n        "loaded_at": datetime.now(timezone.utc).isoformat(),\n    }\n\n    doc_id = generate_doc_id(str(path.resolve()))\n    logger.info("Loaded document \'%s\' (%d chars)", path.name, len(text))\n    return Document(doc_id=doc_id, source=str(path), text=text, metadata=metadata)\n\n\ndef load_documents(paths: List[str | Path]) -> List[Document]:\n    """Load multiple documents, skipping (and logging) any that fail to parse."""\n    documents: List[Document] = []\n    for p in paths:\n        try:\n            documents.append(load_document(p))\n        except DocumentLoadError as exc:\n            logger.warning("Skipping document: %s", exc)\n    return documents\n\n\ndef discover_documents(directory: str | Path) -> List[Path]:\n    """Recursively find all supported document files under `directory`."""\n    directory = Path(directory)\n    if not directory.exists():\n        return []\n    return sorted(\n        p for p in directory.rglob("*") if p.suffix.lower() in SUPPORTED_EXTENSIONS and p.is_file()\n    )\n'

with open(r"03_Multi_Document_Research_Agent/src/document_loader.py", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/src/document_loader.py")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent/src", exist_ok=True)

_content = '"""\nchunker.py\n----------\nSplits loaded documents into overlapping chunks suitable for embedding.\n\nTwo strategies are supported:\n    * "recursive" - splits on a hierarchy of separators (paragraphs, then\n      sentences, then words) trying to keep chunks close to `chunk_size`\n      characters, with `chunk_overlap` characters shared between neighbours.\n    * "semantic"  - splits on sentence boundaries first, then greedily packs\n      whole sentences into chunks until the target size is reached, which\n      keeps sentences intact (never splits mid-sentence).\n"""\n\nfrom __future__ import annotations\n\nimport re\nfrom dataclasses import dataclass, field\nfrom typing import Any, Dict, List\n\nfrom src.document_loader import Document\nfrom src.logger import get_logger\nfrom src.utils import generate_chunk_id\n\nlogger = get_logger(__name__)\n\n_SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\\s+")\n_SEPARATORS = ["\\n\\n", "\\n", ". ", " ", ""]\n\n\n@dataclass\nclass Chunk:\n    """A single chunk of text ready for embedding."""\n\n    chunk_id: str\n    doc_id: str\n    text: str\n    chunk_index: int\n    metadata: Dict[str, Any] = field(default_factory=dict)\n\n\ndef _split_recursive_raw(text: str, chunk_size: int,\n                          separators: List[str] | None = None) -> List[str]:\n    """Recursively split text into non-overlapping pieces using a hierarchy of\n    separators. Falls back to a hard character split if no separator produces\n    small enough pieces (mirrors the behaviour of LangChain\'s\n    RecursiveCharacterTextSplitter, reimplemented here to avoid a hard\n    dependency).\n\n    Overlap is intentionally NOT applied here — it is applied exactly once,\n    by the top-level `_split_recursive` wrapper, to avoid compounding overlap\n    at every level of recursion.\n    """\n    separators = separators or _SEPARATORS\n    if len(text) <= chunk_size:\n        return [text] if text.strip() else []\n\n    separator = separators[0]\n    remaining_separators = separators[1:]\n\n    if separator == "":\n        pieces = [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]\n    else:\n        pieces = text.split(separator)\n\n    chunks: List[str] = []\n    current = ""\n    for piece in pieces:\n        candidate = f"{current}{separator}{piece}" if current else piece\n        if len(candidate) <= chunk_size:\n            current = candidate\n        else:\n            if current:\n                chunks.append(current)\n            if len(piece) > chunk_size and remaining_separators:\n                chunks.extend(_split_recursive_raw(piece, chunk_size, remaining_separators))\n                current = ""\n            else:\n                current = piece\n\n    if current.strip():\n        chunks.append(current)\n\n    return chunks\n\n\ndef _split_recursive(text: str, chunk_size: int, chunk_overlap: int,\n                      separators: List[str] | None = None) -> List[str]:\n    """Split `text` into overlapping chunks: structural splitting via\n    `_split_recursive_raw`, followed by a single pass of overlap injection.\n    """\n    raw_chunks = _split_recursive_raw(text, chunk_size, separators)\n    return _apply_overlap(raw_chunks, chunk_overlap)\n\n\ndef _apply_overlap(chunks: List[str], chunk_overlap: int) -> List[str]:\n    """Prepend a trailing slice of the previous chunk to create overlap."""\n    if chunk_overlap <= 0 or len(chunks) < 2:\n        return chunks\n    overlapped = [chunks[0]]\n    for prev, curr in zip(chunks, chunks[1:]):\n        tail = prev[-chunk_overlap:]\n        overlapped.append(f"{tail} {curr}".strip())\n    return overlapped\n\n\ndef _split_semantic(text: str, chunk_size: int, chunk_overlap: int) -> List[str]:\n    """Pack whole sentences into chunks, never splitting mid-sentence."""\n    sentences = [s.strip() for s in _SENTENCE_SPLIT_RE.split(text) if s.strip()]\n    if not sentences:\n        return []\n\n    chunks: List[str] = []\n    current_sentences: List[str] = []\n    current_len = 0\n\n    for sentence in sentences:\n        projected = current_len + len(sentence) + 1\n        if projected > chunk_size and current_sentences:\n            chunks.append(" ".join(current_sentences))\n            # start next chunk with overlap: carry over trailing sentences\n            overlap_sentences: List[str] = []\n            overlap_len = 0\n            for s in reversed(current_sentences):\n                if overlap_len + len(s) > chunk_overlap:\n                    break\n                overlap_sentences.insert(0, s)\n                overlap_len += len(s)\n            current_sentences = overlap_sentences\n            current_len = overlap_len\n\n        current_sentences.append(sentence)\n        current_len += len(sentence) + 1\n\n    if current_sentences:\n        chunks.append(" ".join(current_sentences))\n\n    return chunks\n\n\nclass Chunker:\n    """Configurable document chunker.\n\n    Args:\n        chunk_size: Target chunk size in characters.\n        chunk_overlap: Overlap (in characters) between consecutive chunks.\n        strategy: "recursive" or "semantic".\n    """\n\n    def __init__(self, chunk_size: int = 800, chunk_overlap: int = 120,\n                 strategy: str = "recursive") -> None:\n        if chunk_overlap >= chunk_size:\n            raise ValueError("chunk_overlap must be smaller than chunk_size")\n        if strategy not in {"recursive", "semantic"}:\n            raise ValueError("strategy must be \'recursive\' or \'semantic\'")\n        self.chunk_size = chunk_size\n        self.chunk_overlap = chunk_overlap\n        self.strategy = strategy\n\n    def split_text(self, text: str) -> List[str]:\n        """Split a raw string into a list of chunk strings using the configured strategy."""\n        if self.strategy == "semantic":\n            return _split_semantic(text, self.chunk_size, self.chunk_overlap)\n        return _split_recursive(text, self.chunk_size, self.chunk_overlap)\n\n    def split_document(self, document: Document) -> List[Chunk]:\n        """Split a single `Document` into a list of `Chunk` objects with inherited metadata."""\n        pieces = self.split_text(document.text)\n        chunks = []\n        for idx, piece in enumerate(pieces):\n            chunk_id = generate_chunk_id(document.doc_id, idx)\n            metadata = {\n                **document.metadata,\n                "doc_id": document.doc_id,\n                "source": document.source,\n                "chunk_index": idx,\n                "num_chunks": len(pieces),\n            }\n            chunks.append(Chunk(chunk_id=chunk_id, doc_id=document.doc_id, text=piece,\n                                 chunk_index=idx, metadata=metadata))\n        logger.info("Split document \'%s\' into %d chunks (%s strategy)",\n                    document.metadata.get("filename", document.doc_id), len(chunks), self.strategy)\n        return chunks\n\n    def split_documents(self, documents: List[Document]) -> List[Chunk]:\n        """Split multiple documents, returning a flat list of chunks."""\n        all_chunks: List[Chunk] = []\n        for doc in documents:\n            all_chunks.extend(self.split_document(doc))\n        return all_chunks\n'

with open(r"03_Multi_Document_Research_Agent/src/chunker.py", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/src/chunker.py")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent/src", exist_ok=True)

_content = '"""\nembedder.py\n-----------\nWraps a SentenceTransformers model to produce dense embeddings for chunks\nand queries. Embeddings are L2-normalised so that inner-product search in\nFAISS is equivalent to cosine similarity.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import List\n\nimport numpy as np\n\nfrom src.config import config\nfrom src.logger import get_logger\n\nlogger = get_logger(__name__)\n\n\nclass Embedder:\n    """Sentence embedding model wrapper.\n\n    Args:\n        model_name: Name of the SentenceTransformers model to load.\n    """\n\n    def __init__(self, model_name: str | None = None) -> None:\n        from sentence_transformers import SentenceTransformer\n\n        self.model_name = model_name or config.embedding_model\n        logger.info("Loading embedding model \'%s\'...", self.model_name)\n        self._model = SentenceTransformer(self.model_name)\n        # `get_sentence_embedding_dimension` was renamed to `get_embedding_dimension`\n        # in newer sentence-transformers releases; support both so this works\n        # regardless of which version is installed.\n        if hasattr(self._model, "get_embedding_dimension"):\n            self.dimension = self._model.get_embedding_dimension()\n        else:  # pragma: no cover - older sentence-transformers versions\n            self.dimension = self._model.get_sentence_embedding_dimension()\n        logger.info("Embedding model loaded (dim=%d)", self.dimension)\n\n    def embed_texts(self, texts: List[str], batch_size: int = 32,\n                     show_progress: bool = False) -> np.ndarray:\n        """Embed a batch of texts, returning an (N, dim) L2-normalised float32 array."""\n        if not texts:\n            return np.zeros((0, self.dimension), dtype="float32")\n        embeddings = self._model.encode(\n            texts,\n            batch_size=batch_size,\n            show_progress_bar=show_progress,\n            normalize_embeddings=True,\n            convert_to_numpy=True,\n        )\n        return embeddings.astype("float32")\n\n    def embed_query(self, query: str) -> np.ndarray:\n        """Embed a single query string, returning a (dim,) L2-normalised float32 vector."""\n        return self.embed_texts([query])[0]\n'

with open(r"03_Multi_Document_Research_Agent/src/embedder.py", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/src/embedder.py")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent/src", exist_ok=True)

_content = '"""\nvector_store.py\n----------------\nFAISS-backed vector database with persistence and incremental indexing.\n\nDesign notes:\n    * Uses `IndexFlatIP` (inner product) over L2-normalised vectors, which is\n      mathematically equivalent to cosine similarity search.\n    * Chunk text + metadata are kept in a parallel Python list (`self.records`)\n      and persisted alongside the FAISS index as JSON, since FAISS itself\n      only stores vectors.\n    * `add()` supports incremental indexing: new chunks are appended to the\n      existing index without rebuilding it from scratch.\n"""\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional\n\nimport numpy as np\n\nfrom src.chunker import Chunk\nfrom src.logger import get_logger\n\nlogger = get_logger(__name__)\n\n_INDEX_FILE = "index.faiss"\n_RECORDS_FILE = "records.json"\n\n\nclass VectorStore:\n    """A persistent, incrementally-updatable FAISS vector store."""\n\n    def __init__(self, dimension: int) -> None:\n        import faiss\n\n        self._faiss = faiss\n        self.dimension = dimension\n        self.index = faiss.IndexFlatIP(dimension)\n        self.records: List[Dict[str, Any]] = []  # parallel array: records[i] <-> index vector i\n\n    def add(self, chunks: List[Chunk], embeddings: np.ndarray) -> None:\n        """Add new chunks and their embeddings to the index (incremental)."""\n        if len(chunks) != embeddings.shape[0]:\n            raise ValueError("Number of chunks must match number of embedding rows")\n        if embeddings.shape[0] == 0:\n            return\n\n        existing_ids = {r["chunk_id"] for r in self.records}\n        new_chunks, new_vectors = [], []\n        for chunk, vector in zip(chunks, embeddings):\n            if chunk.chunk_id in existing_ids:\n                continue  # avoid duplicate indexing on repeated ingestion\n            new_chunks.append(chunk)\n            new_vectors.append(vector)\n\n        if not new_chunks:\n            logger.info("No new chunks to add (all already indexed).")\n            return\n\n        vectors = np.vstack(new_vectors).astype("float32")\n        self.index.add(vectors)\n        for chunk in new_chunks:\n            self.records.append({\n                "chunk_id": chunk.chunk_id,\n                "doc_id": chunk.doc_id,\n                "text": chunk.text,\n                "metadata": chunk.metadata,\n            })\n        logger.info("Added %d new vectors (total=%d)", len(new_chunks), self.index.ntotal)\n\n    def search(self, query_embedding: np.ndarray, top_k: int = 5,\n               metadata_filter: Optional[Dict[str, Any]] = None) -> List[Dict[str, Any]]:\n        """Search the index for the `top_k` most similar chunks.\n\n        Args:\n            query_embedding: (dim,) L2-normalised query vector.\n            top_k: Number of results to return.\n            metadata_filter: Optional dict of exact-match metadata constraints,\n                              e.g. {"file_type": "pdf"}.\n\n        Returns:\n            A list of result dicts with keys: chunk_id, doc_id, text, metadata, score.\n        """\n        if self.index.ntotal == 0:\n            return []\n\n        # Over-fetch when filtering so we still end up with `top_k` after filtering.\n        fetch_k = top_k * 5 if metadata_filter else top_k\n        fetch_k = min(fetch_k, self.index.ntotal)\n\n        query_vector = query_embedding.reshape(1, -1).astype("float32")\n        scores, indices = self.index.search(query_vector, fetch_k)\n\n        results = []\n        for score, idx in zip(scores[0], indices[0]):\n            if idx == -1:\n                continue\n            record = self.records[idx]\n            if metadata_filter and not _matches_filter(record["metadata"], metadata_filter):\n                continue\n            results.append({**record, "score": float(score)})\n            if len(results) >= top_k:\n                break\n        return results\n\n    def save(self, directory: str | Path) -> None:\n        """Persist the FAISS index and metadata records to `directory`."""\n        directory = Path(directory)\n        directory.mkdir(parents=True, exist_ok=True)\n        self._faiss.write_index(self.index, str(directory / _INDEX_FILE))\n        with open(directory / _RECORDS_FILE, "w", encoding="utf-8") as fh:\n            json.dump({"dimension": self.dimension, "records": self.records}, fh)\n        logger.info("Vector store saved to %s (%d vectors)", directory, self.index.ntotal)\n\n    @classmethod\n    def load(cls, directory: str | Path) -> "VectorStore":\n        """Load a previously persisted vector store from `directory`."""\n        import faiss\n\n        directory = Path(directory)\n        index_path = directory / _INDEX_FILE\n        records_path = directory / _RECORDS_FILE\n        if not index_path.exists() or not records_path.exists():\n            raise FileNotFoundError(f"No vector store found at {directory}")\n\n        with open(records_path, "r", encoding="utf-8") as fh:\n            data = json.load(fh)\n\n        store = cls(dimension=data["dimension"])\n        store.index = faiss.read_index(str(index_path))\n        store.records = data["records"]\n        logger.info("Vector store loaded from %s (%d vectors)", directory, store.index.ntotal)\n        return store\n\n    @classmethod\n    def exists(cls, directory: str | Path) -> bool:\n        directory = Path(directory)\n        return (directory / _INDEX_FILE).exists() and (directory / _RECORDS_FILE).exists()\n\n    def __len__(self) -> int:\n        return self.index.ntotal\n\n\ndef _matches_filter(metadata: Dict[str, Any], filter_dict: Dict[str, Any]) -> bool:\n    return all(metadata.get(k) == v for k, v in filter_dict.items())\n'

with open(r"03_Multi_Document_Research_Agent/src/vector_store.py", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/src/vector_store.py")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent/src", exist_ok=True)

_content = '"""\nreranker.py\n-----------\nLexical (BM25) indexing and rank-fusion utilities used to build hybrid\n(vector + keyword) search on top of the FAISS vector store.\n\nRank fusion uses Reciprocal Rank Fusion (RRF), a simple, parameter-light\nmethod that combines ranked lists from different retrieval systems without\nrequiring their scores to be on the same scale.\n"""\n\nfrom __future__ import annotations\n\nimport re\nfrom typing import Any, Dict, List\n\nfrom rank_bm25 import BM25Okapi\n\nfrom src.logger import get_logger\n\nlogger = get_logger(__name__)\n\n_TOKEN_RE = re.compile(r"[A-Za-z0-9\']+")\n\n\ndef _tokenize(text: str) -> List[str]:\n    return [t.lower() for t in _TOKEN_RE.findall(text)]\n\n\nclass BM25Index:\n    """A simple BM25 lexical index over a corpus of chunk records."""\n\n    def __init__(self) -> None:\n        self._records: List[Dict[str, Any]] = []\n        self._bm25: BM25Okapi | None = None\n\n    def build(self, records: List[Dict[str, Any]]) -> None:\n        """(Re)build the BM25 index from a list of chunk record dicts (with a \'text\' key)."""\n        self._records = records\n        tokenized_corpus = [_tokenize(r["text"]) for r in records]\n        self._bm25 = BM25Okapi(tokenized_corpus) if tokenized_corpus else None\n        logger.info("BM25 index built over %d chunks", len(records))\n\n    def search(self, query: str, top_k: int = 10) -> List[Dict[str, Any]]:\n        """Return the top-k records by BM25 score for `query`."""\n        if self._bm25 is None or not self._records:\n            return []\n        scores = self._bm25.get_scores(_tokenize(query))\n        ranked = sorted(zip(self._records, scores), key=lambda pair: pair[1], reverse=True)\n        return [{**record, "score": float(score)} for record, score in ranked[:top_k]]\n\n\ndef reciprocal_rank_fusion(\n    ranked_lists: List[List[Dict[str, Any]]],\n    weights: List[float] | None = None,\n    key: str = "chunk_id",\n    rrf_k: int = 60,\n) -> List[Dict[str, Any]]:\n    """Fuse multiple ranked result lists into a single ranking via weighted RRF.\n\n    Each item\'s fused score is: sum_over_lists( weight_i / (rrf_k + rank_in_list_i) ).\n    Items are deduplicated by `key`, keeping the richest record seen.\n\n    Args:\n        ranked_lists: One ranked list of record dicts per retrieval system.\n        weights: Optional per-list weight (defaults to 1.0 for every list).\n        key: Field used to identify duplicate records across lists.\n        rrf_k: RRF damping constant (60 is the commonly used default).\n\n    Returns:\n        A single list of records sorted by fused score (descending), each\n        record annotated with a `fused_score` field.\n    """\n    weights = weights or [1.0] * len(ranked_lists)\n    fused_scores: Dict[str, float] = {}\n    record_lookup: Dict[str, Dict[str, Any]] = {}\n\n    for weight, ranked_list in zip(weights, ranked_lists):\n        for rank, record in enumerate(ranked_list):\n            rid = record[key]\n            fused_scores[rid] = fused_scores.get(rid, 0.0) + weight / (rrf_k + rank + 1)\n            record_lookup.setdefault(rid, record)\n\n    fused = [\n        {**record_lookup[rid], "fused_score": score}\n        for rid, score in fused_scores.items()\n    ]\n    fused.sort(key=lambda r: r["fused_score"], reverse=True)\n    return fused\n'

with open(r"03_Multi_Document_Research_Agent/src/reranker.py", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/src/reranker.py")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent/src", exist_ok=True)

_content = '"""\nretriever.py\n------------\nThe `HybridRetriever` ties together vector search (FAISS) and lexical\nsearch (BM25), fuses their rankings, removes near-duplicate chunks, applies\nmetadata filtering, and performs light context compression so the final\ncontext passed to the LLM is compact and non-redundant.\n"""\n\nfrom __future__ import annotations\n\nfrom difflib import SequenceMatcher\nfrom typing import Any, Dict, List, Optional\n\nfrom src.config import config\nfrom src.embedder import Embedder\nfrom src.logger import get_logger\nfrom src.reranker import BM25Index, reciprocal_rank_fusion\nfrom src.utils import timer\nfrom src.vector_store import VectorStore\n\nlogger = get_logger(__name__)\n\n\ndef _is_near_duplicate(a: str, b: str, threshold: float = 0.9) -> bool:\n    """Cheap near-duplicate detection using difflib\'s SequenceMatcher ratio."""\n    if abs(len(a) - len(b)) / max(len(a), len(b), 1) > 0.3:\n        return False\n    return SequenceMatcher(None, a, b).ratio() >= threshold\n\n\ndef deduplicate(records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:\n    """Remove exact and near-duplicate chunks, keeping the highest-scored copy."""\n    kept: List[Dict[str, Any]] = []\n    for record in records:\n        if any(_is_near_duplicate(record["text"], k["text"]) for k in kept):\n            continue\n        kept.append(record)\n    return kept\n\n\ndef compress_context(records: List[Dict[str, Any]], max_chars: int) -> List[Dict[str, Any]]:\n    """Trim the list of retrieved records so their combined text fits `max_chars`.\n\n    Greedily keeps the highest-ranked records first, truncating the last one\n    that would overflow the budget rather than dropping it entirely.\n    """\n    compressed: List[Dict[str, Any]] = []\n    used = 0\n    for record in records:\n        remaining = max_chars - used\n        if remaining <= 0:\n            break\n        text = record["text"]\n        if len(text) > remaining:\n            text = text[:remaining].rsplit(" ", 1)[0] + "…"\n        compressed.append({**record, "text": text})\n        used += len(text)\n    return compressed\n\n\nclass HybridRetriever:\n    """Combines vector similarity search and BM25 lexical search.\n\n    Args:\n        vector_store: A populated `VectorStore`.\n        embedder: The `Embedder` used to embed queries.\n        alpha: Weight applied to the vector-search ranking in fusion\n               (1 - alpha is applied to BM25).\n    """\n\n    def __init__(self, vector_store: VectorStore, embedder: Embedder,\n                 alpha: float | None = None) -> None:\n        self.vector_store = vector_store\n        self.embedder = embedder\n        self.alpha = alpha if alpha is not None else config.hybrid_alpha\n        self.bm25_index = BM25Index()\n        self.bm25_index.build(vector_store.records)\n\n    def refresh_bm25(self) -> None:\n        """Rebuild the BM25 index — call after `vector_store.add(...)`."""\n        self.bm25_index.build(self.vector_store.records)\n\n    def retrieve(\n        self,\n        query: str,\n        top_k: Optional[int] = None,\n        metadata_filter: Optional[Dict[str, Any]] = None,\n        max_context_chars: Optional[int] = None,\n    ) -> Dict[str, Any]:\n        """Run hybrid retrieval for `query`, returning ranked, deduplicated,\n        compressed chunks along with timing information.\n\n        Returns:\n            {\n              "chunks": [...],       # final list of chunk records w/ scores\n              "timing": {"vector_search_s": .., "bm25_search_s": .., "total_s": ..},\n            }\n        """\n        top_k = top_k or config.top_k\n        fetch_k = max(top_k * 3, 10)\n        timing: Dict[str, float] = {}\n\n        with timer() as t_total:\n            with timer() as t_vec:\n                query_embedding = self.embedder.embed_query(query)\n                vector_results = self.vector_store.search(\n                    query_embedding, top_k=fetch_k, metadata_filter=metadata_filter\n                )\n            timing["vector_search_s"] = t_vec["elapsed_seconds"]\n\n            with timer() as t_bm25:\n                bm25_results = self.bm25_index.search(query, top_k=fetch_k)\n                if metadata_filter:\n                    bm25_results = [\n                        r for r in bm25_results\n                        if all(r["metadata"].get(k) == v for k, v in metadata_filter.items())\n                    ]\n            timing["bm25_search_s"] = t_bm25["elapsed_seconds"]\n\n            fused = reciprocal_rank_fusion(\n                [vector_results, bm25_results],\n                weights=[self.alpha, 1 - self.alpha],\n            )\n            deduped = deduplicate(fused)[:top_k]\n\n            max_chars = max_context_chars or config.max_context_tokens * 4  # ~4 chars/token\n            final_chunks = compress_context(deduped, max_chars)\n\n        timing["total_s"] = t_total["elapsed_seconds"]\n        logger.info("Retrieved %d chunks for query in %.3fs", len(final_chunks), timing["total_s"])\n        return {"chunks": final_chunks, "timing": timing}\n\n\ndef format_sources(chunks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:\n    """Produce a compact, citation-friendly summary of retrieved chunks."""\n    sources = []\n    for i, chunk in enumerate(chunks, start=1):\n        meta = chunk.get("metadata", {})\n        sources.append({\n            "citation": f"[{i}]",\n            "filename": meta.get("filename", "unknown"),\n            "chunk_index": meta.get("chunk_index"),\n            "score": round(chunk.get("fused_score", chunk.get("score", 0.0)), 4),\n        })\n    return sources\n'

with open(r"03_Multi_Document_Research_Agent/src/retriever.py", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/src/retriever.py")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent/src", exist_ok=True)

_content = '"""\nmemory.py\n---------\nLightweight in-memory conversation history for the RAG pipeline. Keeps the\nlast N turns and can format them for inclusion in prompts (query rewriting,\nfollow-up questions, etc).\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom datetime import datetime, timezone\nfrom typing import Any, Dict, List\n\n\n@dataclass\nclass Turn:\n    """A single question/answer turn in a conversation."""\n\n    query: str\n    answer: str\n    sources: List[Dict[str, Any]] = field(default_factory=list)\n    timestamp: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())\n\n\nclass ConversationMemory:\n    """Fixed-size sliding-window conversation memory.\n\n    Args:\n        max_turns: Maximum number of turns to retain. Older turns are\n                   dropped once this limit is exceeded.\n    """\n\n    def __init__(self, max_turns: int = 6) -> None:\n        self.max_turns = max_turns\n        self._turns: List[Turn] = []\n\n    def add_turn(self, query: str, answer: str, sources: List[Dict[str, Any]] | None = None) -> None:\n        """Record a new conversation turn."""\n        self._turns.append(Turn(query=query, answer=answer, sources=sources or []))\n        if len(self._turns) > self.max_turns:\n            self._turns.pop(0)\n\n    def get_history(self) -> List[Turn]:\n        """Return the retained turns, oldest first."""\n        return list(self._turns)\n\n    def format_for_prompt(self, max_turns: int | None = None) -> str:\n        """Render recent history as plain text for inclusion in an LLM prompt."""\n        turns = self._turns[-max_turns:] if max_turns else self._turns\n        if not turns:\n            return ""\n        lines = []\n        for turn in turns:\n            lines.append(f"User: {turn.query}")\n            lines.append(f"Assistant: {turn.answer}")\n        return "\\n".join(lines)\n\n    def clear(self) -> None:\n        """Discard all stored turns."""\n        self._turns.clear()\n\n    def __len__(self) -> int:\n        return len(self._turns)\n'

with open(r"03_Multi_Document_Research_Agent/src/memory.py", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/src/memory.py")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent/src", exist_ok=True)

_content = '"""\nprompts.py\n----------\nPrompt templates for the RAG pipeline: the main answer-generation prompt,\nand a smaller prompt used for conversational query rewriting (resolving\npronouns / follow-up references against chat history).\n"""\n\nfrom __future__ import annotations\n\nfrom typing import Any, Dict, List\n\nSYSTEM_PROMPT = """You are a meticulous research assistant. Answer the user\'s \\\nquestion using ONLY the information in the provided context excerpts. \\\nCite sources inline using the bracketed numbers, e.g. [1], [2]. \\\nIf the context does not contain enough information to answer confidently, \\\nsay so plainly instead of guessing. Keep answers concise and well-structured."""\n\n\nQUERY_REWRITE_SYSTEM_PROMPT = """You rewrite follow-up questions into fully \\\nself-contained search queries. Given the recent conversation and a new user \\\nmessage, produce ONE rewritten query that resolves pronouns and implicit \\\nreferences (e.g. "it", "that", "the second one") using the conversation \\\ncontext. If the message is already self-contained, return it unchanged. \\\nRespond with ONLY the rewritten query, no explanation."""\n\n\ndef build_context_block(chunks: List[Dict[str, Any]]) -> str:\n    """Render retrieved chunks as a numbered context block for the prompt."""\n    lines = []\n    for i, chunk in enumerate(chunks, start=1):\n        filename = chunk.get("metadata", {}).get("filename", "unknown source")\n        lines.append(f"[{i}] (source: {filename})\\n{chunk[\'text\']}")\n    return "\\n\\n".join(lines)\n\n\ndef build_rag_prompt(query: str, chunks: List[Dict[str, Any]], history: str = "") -> str:\n    """Assemble the full RAG generation prompt from context chunks and history.\n\n    Args:\n        query: The (possibly rewritten) user question.\n        chunks: Retrieved context chunk records.\n        history: Optional formatted conversation history.\n\n    Returns:\n        A single prompt string, ready to send to the Gemini API.\n    """\n    context_block = build_context_block(chunks)\n    history_block = f"\\nConversation so far:\\n{history}\\n" if history else ""\n    return (\n        f"{SYSTEM_PROMPT}\\n\\n"\n        f"Context:\\n{context_block}\\n"\n        f"{history_block}\\n"\n        f"Question: {query}\\n\\n"\n        f"Answer:"\n    )\n\n\ndef build_query_rewrite_prompt(query: str, history: str) -> str:\n    """Assemble the prompt used to rewrite a follow-up query into a standalone one."""\n    return (\n        f"{QUERY_REWRITE_SYSTEM_PROMPT}\\n\\n"\n        f"Conversation:\\n{history or \'(no prior turns)\'}\\n\\n"\n        f"New message: {query}\\n\\n"\n        f"Rewritten query:"\n    )\n'

with open(r"03_Multi_Document_Research_Agent/src/prompts.py", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/src/prompts.py")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent/src", exist_ok=True)

_content = '"""\ngemini_client.py\n----------------\nThin, resilient wrapper around the `google-genai` SDK (the officially\nsupported, unified SDK that replaced the deprecated `google-generativeai`\npackage). Centralises API-key configuration, retry/backoff on transient\nerrors, and exposes both blocking and streaming generation.\n"""\n\nfrom __future__ import annotations\n\nimport time\nfrom typing import Iterator, Optional\n\nfrom src.config import config\nfrom src.logger import get_logger\n\nlogger = get_logger(__name__)\n\n\nclass GeminiClientError(Exception):\n    """Raised when the Gemini API call ultimately fails after retries."""\n\n\nclass GeminiClient:\n    """Wraps `google.genai` for text generation.\n\n    Args:\n        api_key: Gemini API key. Falls back to `config.gemini_api_key`.\n        model_name: Model to use, e.g. "gemini-3.5-flash".\n        temperature: Sampling temperature.\n    """\n\n    def __init__(self, api_key: Optional[str] = None, model_name: Optional[str] = None,\n                 temperature: Optional[float] = None) -> None:\n        from google import genai\n\n        self.api_key = api_key or config.gemini_api_key\n        if not self.api_key or self.api_key == "PASTE_YOUR_API_KEY_HERE":\n            logger.warning(\n                "No valid Gemini API key configured. Set GEMINI_API_KEY before generating."\n            )\n\n        self._genai = genai\n        self._client = genai.Client(api_key=self.api_key)\n        self.model_name = model_name or config.gemini_model\n        self.temperature = temperature if temperature is not None else config.temperature\n        self.min_interval = config.min_request_interval_s\n        self._last_call_ts = 0.0\n\n    def _throttle(self) -> None:\n        \"\"\"Sleep just long enough to respect min_request_interval_s between calls.\n\n        Free-tier Gemini keys are capped at a handful of requests per minute,\n        so a naive loop of back-to-back calls trips a 429 almost immediately.\n        \"\"\"\n        elapsed = time.monotonic() - self._last_call_ts\n        if elapsed < self.min_interval:\n            time.sleep(self.min_interval - elapsed)\n        self._last_call_ts = time.monotonic()\n\n    def generate(self, prompt: str, max_retries: Optional[int] = None) -> str:\n        """Generate a complete (non-streaming) response for `prompt`.\n\n        Retries transient failures with exponential backoff.\n        """\n        from google.genai import types\n\n        max_retries = max_retries if max_retries is not None else config.max_retries\n        last_error: Exception | None = None\n        self._throttle()\n\n        for attempt in range(1, max_retries + 1):\n            try:\n                response = self._client.models.generate_content(\n                    model=self.model_name,\n                    contents=prompt,\n                    config=types.GenerateContentConfig(temperature=self.temperature),\n                )\n                return (response.text or "").strip()\n            except Exception as exc:  # noqa: BLE001 - broad by design, then re-raised\n                last_error = exc\n                wait = min(2 ** attempt, 20)\n                logger.warning("Gemini generate() attempt %d/%d failed: %s. Retrying in %ds",\n                               attempt, max_retries, exc, wait)\n                time.sleep(wait)\n\n        raise GeminiClientError(f"Gemini generation failed after {max_retries} attempts: {last_error}")\n\n    def generate_stream(self, prompt: str) -> Iterator[str]:\n        """Yield response text incrementally as it is generated.\n\n        On failure, yields a single error message rather than raising, so\n        streaming consumers (CLI/Streamlit) can display it gracefully.\n        """\n        from google.genai import types\n\n        try:\n            self._throttle()\n            stream = self._client.models.generate_content_stream(\n                model=self.model_name,\n                contents=prompt,\n                config=types.GenerateContentConfig(temperature=self.temperature),\n            )\n            for chunk in stream:\n                if chunk.text:\n                    yield chunk.text\n        except Exception as exc:  # noqa: BLE001\n            logger.error("Gemini streaming generation failed: %s", exc)\n            yield f"\\n[Error: generation failed — {exc}]"\n'

with open(r"03_Multi_Document_Research_Agent/src/gemini_client.py", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/src/gemini_client.py")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent/src", exist_ok=True)

_content = '"""\npipeline.py\n-----------\n`RAGPipeline` is the top-level orchestrator that wires together document\ningestion, chunking, embedding, hybrid retrieval, prompt construction, and\nGemini generation. This is the single entry point used by the CLI, the\nFastAPI service, and the Streamlit app.\n"""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional\n\nfrom src.chunker import Chunker\nfrom src.config import AppConfig, config as default_config\nfrom src.document_loader import discover_documents, load_documents\nfrom src.embedder import Embedder\nfrom src.gemini_client import GeminiClient\nfrom src.logger import get_logger\nfrom src.memory import ConversationMemory\nfrom src.prompts import build_query_rewrite_prompt, build_rag_prompt\nfrom src.retriever import HybridRetriever, format_sources\nfrom src.utils import timer\nfrom src.vector_store import VectorStore\n\nlogger = get_logger(__name__)\n\n\nclass RAGPipeline:\n    """End-to-end multi-document RAG pipeline.\n\n    Args:\n        cfg: An `AppConfig`. Defaults to the module-level singleton config.\n        lazy_gemini: If True (default), the Gemini client is only created on\n                     first use — useful for ingestion-only workflows / tests\n                     that don\'t need a valid API key.\n    """\n\n    def __init__(self, cfg: Optional[AppConfig] = None, lazy_gemini: bool = True) -> None:\n        self.config = cfg or default_config\n        self.chunker = Chunker(\n            chunk_size=self.config.chunk_size,\n            chunk_overlap=self.config.chunk_overlap,\n            strategy=self.config.chunking_strategy,\n        )\n        self.embedder = Embedder(self.config.embedding_model)\n        self.memory = ConversationMemory()\n\n        if VectorStore.exists(self.config.vectorstore_dir):\n            self.vector_store = VectorStore.load(self.config.vectorstore_dir)\n        else:\n            self.vector_store = VectorStore(dimension=self.embedder.dimension)\n\n        self.retriever = HybridRetriever(self.vector_store, self.embedder, alpha=self.config.hybrid_alpha)\n\n        self._gemini: Optional[GeminiClient] = None if lazy_gemini else self._build_gemini()\n\n    def _build_gemini(self) -> GeminiClient:\n        return GeminiClient(\n            api_key=self.config.gemini_api_key,\n            model_name=self.config.gemini_model,\n            temperature=self.config.temperature,\n        )\n\n    @property\n    def gemini(self) -> GeminiClient:\n        if self._gemini is None:\n            self._gemini = self._build_gemini()\n        return self._gemini\n\n    # ------------------------------------------------------------------ #\n    # Ingestion\n    # ------------------------------------------------------------------ #\n    def ingest_paths(self, paths: List[str]) -> Dict[str, Any]:\n        """Ingest a specific list of file paths into the vector store."""\n        with timer() as t:\n            documents = load_documents(paths)\n            chunks = self.chunker.split_documents(documents)\n            embeddings = self.embedder.embed_texts([c.text for c in chunks], show_progress=True)\n            self.vector_store.add(chunks, embeddings)\n            self.retriever.refresh_bm25()\n            self.vector_store.save(self.config.vectorstore_dir)\n        logger.info("Ingested %d documents / %d chunks in %.2fs",\n                    len(documents), len(chunks), t["elapsed_seconds"])\n        return {\n            "num_documents": len(documents),\n            "num_chunks": len(chunks),\n            "elapsed_seconds": t["elapsed_seconds"],\n            "total_vectors": len(self.vector_store),\n        }\n\n    def ingest_directory(self, directory: Optional[str] = None) -> Dict[str, Any]:\n        """Discover and ingest every supported document under `directory`."""\n        directory = directory or self.config.documents_dir\n        paths = [str(p) for p in discover_documents(directory)]\n        if not paths:\n            logger.warning("No supported documents found in %s", directory)\n            return {"num_documents": 0, "num_chunks": 0, "elapsed_seconds": 0.0, "total_vectors": len(self.vector_store)}\n        return self.ingest_paths(paths)\n\n    # ------------------------------------------------------------------ #\n    # Querying\n    # ------------------------------------------------------------------ #\n    def _rewrite_query(self, query: str) -> str:\n        """Resolve follow-up references in `query` using conversation history."""\n        history = self.memory.format_for_prompt(max_turns=3)\n        if not history:\n            return query\n        try:\n            prompt = build_query_rewrite_prompt(query, history)\n            rewritten = self.gemini.generate(prompt)\n            return rewritten or query\n        except Exception as exc:  # noqa: BLE001 - fall back to original query\n            logger.warning("Query rewriting failed, using original query: %s", exc)\n            return query\n\n    def query(\n        self,\n        user_query: str,\n        top_k: Optional[int] = None,\n        metadata_filter: Optional[Dict[str, Any]] = None,\n        use_memory: bool = True,\n        rewrite_query: Optional[bool] = None,\n        stream: bool = False,\n    ) -> Dict[str, Any]:\n        """Answer a user question via retrieval-augmented generation.\n\n        Returns a dict with: answer (str, or a generator if stream=True),\n        sources, retrieved_chunks, rewritten_query, and timing.\n        """\n        with timer() as t_total:\n            do_rewrite = self.config.enable_query_rewrite if rewrite_query is None else rewrite_query\n            search_query = self._rewrite_query(user_query) if (do_rewrite and use_memory) else user_query\n            retrieval = self.retriever.retrieve(search_query, top_k=top_k, metadata_filter=metadata_filter)\n            chunks = retrieval["chunks"]\n            history = self.memory.format_for_prompt() if use_memory else ""\n            prompt = build_rag_prompt(user_query, chunks, history=history)\n\n            if stream:\n                answer: Any = self.gemini.generate_stream(prompt)\n            else:\n                if not chunks:\n                    answer = (\n                        "I couldn\'t find relevant information in the indexed documents "\n                        "to answer that question."\n                    )\n                else:\n                    try:\n                        answer = self.gemini.generate(prompt)\n                    except Exception as exc:  # noqa: BLE001 - surface a friendly message, not a 500\n                        logger.error("Generation failed: %s", exc)\n                        answer = (\n                            "Sorry, I couldn\'t generate an answer right now — the language "\n                            "model request failed. Check that GEMINI_API_KEY is valid and "\n                            "try again."\n                        )\n                if use_memory:\n                    self.memory.add_turn(user_query, answer, sources=format_sources(chunks))\n\n        return {\n            "query": user_query,\n            "rewritten_query": search_query,\n            "answer": answer,\n            "sources": format_sources(chunks),\n            "retrieved_chunks": chunks,\n            "timing": {**retrieval["timing"], "total_pipeline_s": t_total["elapsed_seconds"]},\n        }\n\n    def reset_memory(self) -> None:\n        self.memory.clear()\n\n    def clear_index(self) -> None:\n        """Wipe the vector store (used by the \'Clear Database\' UI action)."""\n        self.vector_store = VectorStore(dimension=self.embedder.dimension)\n        self.retriever = HybridRetriever(self.vector_store, self.embedder, alpha=self.config.hybrid_alpha)\n        self.vector_store.save(self.config.vectorstore_dir)\n        logger.info("Vector store cleared.")\n'

with open(r"03_Multi_Document_Research_Agent/src/pipeline.py", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/src/pipeline.py")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent/src", exist_ok=True)

_content = '"""\nevaluation.py\n-------------\nRetrieval and pipeline evaluation utilities: Precision@K, Recall@K, latency\nbenchmarking, and a small harness that runs a labelled query set end-to-end.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Any, Dict, List, TypedDict\n\nimport numpy as np\nimport pandas as pd\n\nfrom src.logger import get_logger\nfrom src.utils import timer\n\nlogger = get_logger(__name__)\n\n\nclass LabelledQuery(TypedDict):\n    """A query with a set of known-relevant chunk/doc IDs, for evaluation."""\n\n    query: str\n    relevant_ids: List[str]\n\n\ndef precision_at_k(retrieved_ids: List[str], relevant_ids: List[str], k: int) -> float:\n    """Fraction of the top-k retrieved IDs that are relevant."""\n    if k <= 0:\n        return 0.0\n    top_k = retrieved_ids[:k]\n    if not top_k:\n        return 0.0\n    hits = sum(1 for rid in top_k if rid in relevant_ids)\n    return hits / len(top_k)\n\n\ndef recall_at_k(retrieved_ids: List[str], relevant_ids: List[str], k: int) -> float:\n    """Fraction of all relevant IDs that appear in the top-k retrieved results."""\n    if not relevant_ids:\n        return 0.0\n    top_k = set(retrieved_ids[:k])\n    hits = sum(1 for rid in relevant_ids if rid in top_k)\n    return hits / len(relevant_ids)\n\n\ndef mean_reciprocal_rank(retrieved_ids: List[str], relevant_ids: List[str]) -> float:\n    """1 / rank of the first relevant result, or 0 if none are relevant."""\n    for rank, rid in enumerate(retrieved_ids, start=1):\n        if rid in relevant_ids:\n            return 1.0 / rank\n    return 0.0\n\n\ndef evaluate_retrieval(pipeline: "Any", labelled_queries: List[LabelledQuery],\n                        k: int = 5) -> pd.DataFrame:\n    """Run retrieval-only evaluation over a labelled query set.\n\n    Args:\n        pipeline: A `RAGPipeline` instance (only `.retriever` is used).\n        labelled_queries: Queries with their known-relevant chunk IDs.\n        k: Cutoff for Precision@K / Recall@K.\n\n    Returns:\n        A DataFrame with one row per query and columns:\n        query, precision_at_k, recall_at_k, mrr, latency_s.\n    """\n    rows = []\n    for item in labelled_queries:\n        with timer() as t:\n            result = pipeline.retriever.retrieve(item["query"], top_k=k)\n        retrieved_ids = [c["chunk_id"] for c in result["chunks"]]\n        rows.append({\n            "query": item["query"],\n            "precision_at_k": precision_at_k(retrieved_ids, item["relevant_ids"], k),\n            "recall_at_k": recall_at_k(retrieved_ids, item["relevant_ids"], k),\n            "mrr": mean_reciprocal_rank(retrieved_ids, item["relevant_ids"]),\n            "latency_s": t["elapsed_seconds"],\n        })\n    df = pd.DataFrame(rows)\n    logger.info("Evaluated %d queries — mean P@%d=%.3f, mean R@%d=%.3f",\n                len(df), k, df["precision_at_k"].mean(), k, df["recall_at_k"].mean())\n    return df\n\n\ndef measure_latency(pipeline: "Any", queries: List[str], stream: bool = False) -> pd.DataFrame:\n    """Measure end-to-end query latency (retrieval + generation) for a list of queries."""\n    rows = []\n    for q in queries:\n        result = pipeline.query(q, use_memory=False, rewrite_query=False, stream=stream)\n        timing = result["timing"]\n        rows.append({\n            "query": q,\n            "vector_search_s": timing.get("vector_search_s", 0.0),\n            "bm25_search_s": timing.get("bm25_search_s", 0.0),\n            "retrieval_total_s": timing.get("total_s", 0.0),\n            "pipeline_total_s": timing.get("total_pipeline_s", 0.0),\n        })\n    return pd.DataFrame(rows)\n\n\ndef summarize_latency(df: pd.DataFrame) -> Dict[str, float]:\n    """Compute mean/median/p95 latency summary statistics from `measure_latency` output."""\n    values = df["pipeline_total_s"].to_numpy()\n    if values.size == 0:\n        return {"mean_s": 0.0, "median_s": 0.0, "p95_s": 0.0}\n    return {\n        "mean_s": float(np.mean(values)),\n        "median_s": float(np.median(values)),\n        "p95_s": float(np.percentile(values, 95)),\n    }\n'

with open(r"03_Multi_Document_Research_Agent/src/evaluation.py", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/src/evaluation.py")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent/src", exist_ok=True)

_content = '"""\nvisualization.py\n-----------------\nPlotting helpers for embedding space visualisation and retrieval timing,\nbuilt on Plotly so they render nicely both in Colab/Jupyter and when saved\nas standalone HTML in the Streamlit app.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import List, Optional\n\nimport numpy as np\nimport pandas as pd\nimport plotly.express as px\nimport plotly.graph_objects as go\nfrom sklearn.decomposition import PCA\n\nfrom src.logger import get_logger\n\nlogger = get_logger(__name__)\n\n\ndef plot_embedding_space(embeddings: np.ndarray, labels: List[str],\n                          color_by: Optional[List[str]] = None,\n                          title: str = "Document Chunk Embedding Space (PCA)") -> go.Figure:\n    """Project high-dimensional chunk embeddings into 2D with PCA and scatter-plot them.\n\n    Args:\n        embeddings: (N, dim) array of chunk embeddings.\n        labels: Hover labels (e.g. truncated chunk text) for each point.\n        color_by: Optional categorical value per point (e.g. source filename)\n                  used to colour the scatter plot.\n        title: Chart title.\n\n    Returns:\n        A Plotly `Figure`.\n    """\n    if embeddings.shape[0] < 2:\n        logger.warning("Need at least 2 embeddings to plot; got %d", embeddings.shape[0])\n        return go.Figure()\n\n    n_components = 2\n    pca = PCA(n_components=n_components)\n    projected = pca.fit_transform(embeddings)\n\n    df = pd.DataFrame({\n        "x": projected[:, 0],\n        "y": projected[:, 1],\n        "label": labels,\n        "source": color_by if color_by else ["all"] * len(labels),\n    })\n    fig = px.scatter(\n        df, x="x", y="y", color="source", hover_name="label",\n        title=title, template="plotly_dark",\n    )\n    fig.update_traces(marker=dict(size=9, opacity=0.8, line=dict(width=1, color="white")))\n    return fig\n\n\ndef plot_retrieval_timing(timing_df: pd.DataFrame,\n                           title: str = "Retrieval & Generation Latency by Query") -> go.Figure:\n    """Bar chart comparing vector search / BM25 / total latency across evaluated queries."""\n    fig = go.Figure()\n    if "vector_search_s" in timing_df:\n        fig.add_bar(name="Vector search", x=timing_df.index, y=timing_df["vector_search_s"])\n    if "bm25_search_s" in timing_df:\n        fig.add_bar(name="BM25 search", x=timing_df.index, y=timing_df["bm25_search_s"])\n    if "pipeline_total_s" in timing_df:\n        fig.add_bar(name="Total pipeline", x=timing_df.index, y=timing_df["pipeline_total_s"])\n    fig.update_layout(barmode="group", title=title, template="plotly_dark",\n                       xaxis_title="Query index", yaxis_title="Seconds")\n    return fig\n\n\ndef plot_precision_recall(eval_df: pd.DataFrame,\n                           title: str = "Precision@K / Recall@K by Query") -> go.Figure:\n    """Bar chart of per-query Precision@K and Recall@K from `evaluate_retrieval` output."""\n    fig = go.Figure()\n    fig.add_bar(name="Precision@K", x=eval_df["query"], y=eval_df["precision_at_k"])\n    fig.add_bar(name="Recall@K", x=eval_df["query"], y=eval_df["recall_at_k"])\n    fig.update_layout(barmode="group", title=title, template="plotly_dark",\n                       xaxis_title="Query", yaxis_title="Score", xaxis_tickangle=-30)\n    return fig\n'

with open(r"03_Multi_Document_Research_Agent/src/visualization.py", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/src/visualization.py")


## 7. Generate Interfaces — CLI, FastAPI REST API, Streamlit Chat UI

In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent", exist_ok=True)

_content = '#!/usr/bin/env python3\n"""\nmain.py\n-------\nCommand-line interface for the Multi-Document Research Agent.\n\nUsage:\n    python main.py ingest --path data/documents\n    python main.py query "What is the refund policy?"\n    python main.py chat\n    python main.py evaluate\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport sys\n\nfrom rich.console import Console\nfrom rich.markdown import Markdown\nfrom rich.panel import Panel\nfrom rich.table import Table\n\nfrom src.pipeline import RAGPipeline\n\nconsole = Console()\n\n\ndef cmd_ingest(args: argparse.Namespace) -> None:\n    pipeline = RAGPipeline(lazy_gemini=True)\n    with console.status(f"Ingesting documents from \'{args.path}\'..."):\n        stats = pipeline.ingest_directory(args.path)\n    table = Table(title="Ingestion Summary")\n    table.add_column("Metric")\n    table.add_column("Value")\n    for key, value in stats.items():\n        table.add_row(key, str(value))\n    console.print(table)\n\n\ndef cmd_query(args: argparse.Namespace) -> None:\n    pipeline = RAGPipeline(lazy_gemini=False)\n    with console.status("Thinking..."):\n        result = pipeline.query(args.question, top_k=args.top_k)\n    console.print(Panel(Markdown(result["answer"]), title="Answer", border_style="cyan"))\n\n    if result["sources"]:\n        table = Table(title="Sources")\n        table.add_column("Citation")\n        table.add_column("File")\n        table.add_column("Chunk #")\n        table.add_column("Score")\n        for src in result["sources"]:\n            table.add_row(src["citation"], src["filename"], str(src["chunk_index"]), str(src["score"]))\n        console.print(table)\n\n    console.print(f"[dim]Total time: {result[\'timing\'][\'total_pipeline_s\']:.2f}s[/dim]")\n\n\ndef cmd_chat(args: argparse.Namespace) -> None:\n    pipeline = RAGPipeline(lazy_gemini=False)\n    console.print("[bold cyan]Multi-Document Research Agent — interactive chat[/bold cyan]")\n    console.print("[dim]Type \'exit\' to quit, \'reset\' to clear memory.[/dim]\\n")\n    while True:\n        try:\n            question = console.input("[bold green]You:[/bold green] ")\n        except (EOFError, KeyboardInterrupt):\n            break\n        if question.strip().lower() in {"exit", "quit"}:\n            break\n        if question.strip().lower() == "reset":\n            pipeline.reset_memory()\n            console.print("[dim]Memory cleared.[/dim]")\n            continue\n        with console.status("Thinking..."):\n            result = pipeline.query(question, top_k=args.top_k)\n        console.print(Panel(Markdown(result["answer"]), title="Assistant", border_style="cyan"))\n\n\ndef cmd_evaluate(args: argparse.Namespace) -> None:\n    from src.evaluation import evaluate_retrieval\n\n    pipeline = RAGPipeline(lazy_gemini=True)\n    sample_queries = [\n        {"query": "What is this project about?", "relevant_ids": []},\n    ]\n    df = evaluate_retrieval(pipeline, sample_queries, k=args.top_k)\n    console.print(df.to_string(index=False))\n\n\ndef build_parser() -> argparse.ArgumentParser:\n    parser = argparse.ArgumentParser(prog="research-agent", description="Multi-Document Research Agent CLI")\n    subparsers = parser.add_subparsers(dest="command", required=True)\n\n    p_ingest = subparsers.add_parser("ingest", help="Ingest documents into the vector store")\n    p_ingest.add_argument("--path", default=None, help="Directory of documents to ingest")\n    p_ingest.set_defaults(func=cmd_ingest)\n\n    p_query = subparsers.add_parser("query", help="Ask a single question")\n    p_query.add_argument("question", help="The question to ask")\n    p_query.add_argument("--top-k", type=int, default=5)\n    p_query.set_defaults(func=cmd_query)\n\n    p_chat = subparsers.add_parser("chat", help="Interactive chat session")\n    p_chat.add_argument("--top-k", type=int, default=5)\n    p_chat.set_defaults(func=cmd_chat)\n\n    p_eval = subparsers.add_parser("evaluate", help="Run retrieval evaluation")\n    p_eval.add_argument("--top-k", type=int, default=5)\n    p_eval.set_defaults(func=cmd_evaluate)\n\n    return parser\n\n\ndef main() -> None:\n    parser = build_parser()\n    args = parser.parse_args()\n    args.func(args)\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n'

with open(r"03_Multi_Document_Research_Agent/main.py", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/main.py")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent", exist_ok=True)

_content = '"""\napi.py\n------\nFastAPI REST API for the Multi-Document Research Agent.\n\nEndpoints:\n    GET    /health          - liveness check\n    GET    /config           - current (redacted) configuration\n    POST   /upload           - upload one or more documents and index them\n    POST   /query             - ask a question\n    DELETE /documents/{doc_id} - remove a document\'s chunks (requires reindex)\n    POST   /reindex           - rebuild the vector store from data/documents\n\nRun with:  uvicorn api:app --reload --port 8000\nSwagger UI is auto-generated at /docs.\n"""\n\nfrom __future__ import annotations\n\nimport shutil\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional\n\nfrom fastapi import FastAPI, File, HTTPException, Request, UploadFile\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom fastapi.responses import JSONResponse\nfrom pydantic import BaseModel, Field\n\nfrom src.config import config\nfrom src.logger import get_logger\nfrom src.pipeline import RAGPipeline\n\nlogger = get_logger(__name__)\n\napp = FastAPI(\n    title="Multi-Document Research Agent API",\n    description="Hybrid-search RAG service over PDF / DOCX / TXT documents, powered by Gemini.",\n    version="1.0.0",\n)\n\napp.add_middleware(\n    CORSMiddleware,\n    allow_origins=["*"],\n    allow_methods=["*"],\n    allow_headers=["*"],\n)\n\n\n@app.exception_handler(Exception)\nasync def unhandled_exception_handler(request: Request, exc: Exception) -> JSONResponse:\n    """Catch-all so unexpected errors return clean JSON instead of a raw 500 page."""\n    logger.error("Unhandled error on %s: %s", request.url.path, exc)\n    return JSONResponse(status_code=500, content={"detail": f"Internal server error: {exc}"})\n\n_pipeline: Optional[RAGPipeline] = None\n\n\ndef get_pipeline() -> RAGPipeline:\n    """Lazily instantiate the (expensive-to-load) pipeline as a singleton."""\n    global _pipeline\n    if _pipeline is None:\n        _pipeline = RAGPipeline(lazy_gemini=True)\n    return _pipeline\n\n\nclass QueryRequest(BaseModel):\n    question: str = Field(..., description="The user\'s natural-language question")\n    top_k: int = Field(default=5, ge=1, le=20)\n    metadata_filter: Optional[Dict[str, Any]] = Field(default=None)\n    use_memory: bool = Field(default=True)\n\n\nclass SourceModel(BaseModel):\n    citation: str\n    filename: str\n    chunk_index: Optional[int]\n    score: float\n\n\nclass QueryResponse(BaseModel):\n    answer: str\n    sources: List[SourceModel]\n    rewritten_query: str\n    timing: Dict[str, float]\n\n\nclass UploadResponse(BaseModel):\n    num_documents: int\n    num_chunks: int\n    elapsed_seconds: float\n    total_vectors: int\n\n\nclass HealthResponse(BaseModel):\n    status: str\n    total_vectors: int\n\n\n@app.get("/health", response_model=HealthResponse, tags=["system"])\ndef health() -> HealthResponse:\n    """Liveness/readiness probe."""\n    pipeline = get_pipeline()\n    return HealthResponse(status="ok", total_vectors=len(pipeline.vector_store))\n\n\n@app.get("/config", tags=["system"])\ndef get_config() -> Dict[str, Any]:\n    """Return the current (secret-redacted) configuration."""\n    return config.to_dict()\n\n\n@app.post("/upload", response_model=UploadResponse, tags=["documents"])\nasync def upload_documents(files: List[UploadFile] = File(...)) -> UploadResponse:\n    """Upload one or more documents, save them to `data/documents/`, and index them."""\n    pipeline = get_pipeline()\n    saved_paths = []\n    documents_dir = Path(config.documents_dir)\n    documents_dir.mkdir(parents=True, exist_ok=True)\n\n    for upload in files:\n        suffix = Path(upload.filename).suffix.lower()\n        if suffix not in {".pdf", ".docx", ".txt", ".md"}:\n            raise HTTPException(status_code=400, detail=f"Unsupported file type: {suffix}")\n        dest = documents_dir / upload.filename\n        with open(dest, "wb") as fh:\n            shutil.copyfileobj(upload.file, fh)\n        saved_paths.append(str(dest))\n\n    stats = pipeline.ingest_paths(saved_paths)\n    return UploadResponse(**stats)\n\n\n@app.post("/query", response_model=QueryResponse, tags=["query"])\ndef query(request: QueryRequest) -> QueryResponse:\n    """Ask a question against the indexed document collection."""\n    pipeline = get_pipeline()\n    if len(pipeline.vector_store) == 0:\n        raise HTTPException(status_code=400, detail="No documents indexed yet. Call /upload or /reindex first.")\n\n    result = pipeline.query(\n        request.question,\n        top_k=request.top_k,\n        metadata_filter=request.metadata_filter,\n        use_memory=request.use_memory,\n    )\n    return QueryResponse(\n        answer=result["answer"],\n        sources=[SourceModel(**s) for s in result["sources"]],\n        rewritten_query=result["rewritten_query"],\n        timing=result["timing"],\n    )\n\n\n@app.delete("/documents/{doc_id}", tags=["documents"])\ndef delete_document(doc_id: str) -> Dict[str, str]:\n    """Remove all chunks belonging to `doc_id` and persist the updated index.\n\n    Note: FAISS\'s IndexFlatIP does not support in-place deletion, so this\n    rebuilds the index from the remaining records.\n    """\n    pipeline = get_pipeline()\n    remaining = [r for r in pipeline.vector_store.records if r["doc_id"] != doc_id]\n    if len(remaining) == len(pipeline.vector_store.records):\n        raise HTTPException(status_code=404, detail=f"No chunks found for doc_id={doc_id}")\n\n    from src.vector_store import VectorStore\n    new_store = VectorStore(dimension=pipeline.embedder.dimension)\n    if remaining:\n        embeddings = pipeline.embedder.embed_texts([r["text"] for r in remaining])\n        from src.chunker import Chunk\n        chunks = [Chunk(chunk_id=r["chunk_id"], doc_id=r["doc_id"], text=r["text"],\n                         chunk_index=r["metadata"].get("chunk_index", 0), metadata=r["metadata"])\n                  for r in remaining]\n        new_store.add(chunks, embeddings)\n    new_store.save(config.vectorstore_dir)\n\n    pipeline.vector_store = new_store\n    pipeline.retriever.vector_store = new_store\n    pipeline.retriever.refresh_bm25()\n\n    return {"status": "deleted", "doc_id": doc_id, "remaining_chunks": str(len(remaining))}\n\n\n@app.post("/reindex", response_model=UploadResponse, tags=["documents"])\ndef reindex() -> UploadResponse:\n    """Rebuild the vector store from every document currently in `data/documents/`."""\n    pipeline = get_pipeline()\n    pipeline.clear_index()\n    stats = pipeline.ingest_directory()\n    return UploadResponse(**stats)\n'

with open(r"03_Multi_Document_Research_Agent/api.py", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/api.py")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent", exist_ok=True)

_content = '"""\nstreamlit_app.py\n-----------------\nStreamlit chat UI for the Multi-Document Research Agent.\n\nRun with:  streamlit run streamlit_app.py\n"""\n\nfrom __future__ import annotations\n\nimport time\nfrom pathlib import Path\n\nimport streamlit as st\n\nfrom src.config import config\nfrom src.pipeline import RAGPipeline\n\nst.set_page_config(\n    page_title="Multi-Document Research Agent",\n    page_icon="",\n    layout="wide",\n    initial_sidebar_state="expanded",\n)\n\n# ---------------------------------------------------------------------- #\n# Dark theme styling\n# ---------------------------------------------------------------------- #\nst.markdown(\n    """\n    <style>\n    .stApp { background-color: #0e1117; color: #e6e6e6; }\n    section[data-testid="stSidebar"] { background-color: #131722; }\n    .source-card {\n        background-color: #1a1f2b; border-radius: 8px; padding: 10px 14px;\n        margin-bottom: 8px; border-left: 3px solid #4f8bf9;\n    }\n    </style>\n    """,\n    unsafe_allow_html=True,\n)\n\n\n@st.cache_resource(show_spinner="Loading models & vector store…")\ndef get_pipeline() -> RAGPipeline:\n    return RAGPipeline(lazy_gemini=True)\n\n\nif "chat_history" not in st.session_state:\n    st.session_state.chat_history = []  # list of {"role": ..., "content": ..., "sources": ..., "elapsed": ...}\n\npipeline = get_pipeline()\n\n# ---------------------------------------------------------------------- #\n# Sidebar\n# ---------------------------------------------------------------------- #\nwith st.sidebar:\n    st.title(" Research Agent")\n    st.caption(f"Model: `{config.gemini_model}`  ·  Vectors indexed: `{len(pipeline.vector_store)}`")\n\n    st.subheader(" Upload documents")\n    uploaded_files = st.file_uploader(\n        "PDF, DOCX, or TXT — multiple files supported",\n        type=["pdf", "docx", "txt", "md"],\n        accept_multiple_files=True,\n    )\n    if uploaded_files and st.button("Index uploaded files", use_container_width=True):\n        documents_dir = Path(config.documents_dir)\n        documents_dir.mkdir(parents=True, exist_ok=True)\n        saved_paths = []\n        for uf in uploaded_files:\n            dest = documents_dir / uf.name\n            dest.write_bytes(uf.getbuffer())\n            saved_paths.append(str(dest))\n        with st.spinner("Chunking, embedding, and indexing…"):\n            stats = pipeline.ingest_paths(saved_paths)\n        st.success(f"Indexed {stats[\'num_chunks\']} chunks from {stats[\'num_documents\']} document(s).")\n\n    st.divider()\n    st.subheader(" Retrieval settings")\n    top_k = st.slider("Top-K chunks", min_value=1, max_value=15, value=config.top_k)\n    hybrid_alpha = st.slider("Vector ↔ BM25 balance", min_value=0.0, max_value=1.0,\n                              value=config.hybrid_alpha, step=0.05,\n                              help="1.0 = pure vector search, 0.0 = pure keyword (BM25) search")\n    pipeline.retriever.alpha = hybrid_alpha\n\n    st.divider()\n    col_a, col_b = st.columns(2)\n    with col_a:\n        if st.button(" Reset chat", use_container_width=True):\n            st.session_state.chat_history = []\n            pipeline.reset_memory()\n            st.rerun()\n    with col_b:\n        if st.button(" Clear DB", use_container_width=True):\n            pipeline.clear_index()\n            st.session_state.chat_history = []\n            st.rerun()\n\n# ---------------------------------------------------------------------- #\n# Main chat window\n# ---------------------------------------------------------------------- #\nst.header("Multi-Document Research Agent")\nst.caption("Ask questions across every document you\'ve indexed. Answers are grounded with inline citations.")\n\nfor turn in st.session_state.chat_history:\n    with st.chat_message(turn["role"]):\n        st.markdown(turn["content"])\n        if turn.get("sources"):\n            with st.expander(f" Sources & retrieved chunks · {turn.get(\'elapsed\', 0):.2f}s"):\n                for src in turn["sources"]:\n                    st.markdown(\n                        f"<div class=\'source-card\'><b>{src[\'citation\']} {src[\'filename\']}</b> "\n                        f"— chunk #{src[\'chunk_index\']} · score {src[\'score\']}</div>",\n                        unsafe_allow_html=True,\n                    )\n\nuser_question = st.chat_input("Ask a question about your documents…")\nif user_question:\n    st.session_state.chat_history.append({"role": "user", "content": user_question})\n    with st.chat_message("user"):\n        st.markdown(user_question)\n\n    with st.chat_message("assistant"):\n        if len(pipeline.vector_store) == 0:\n            answer = "No documents are indexed yet — upload some files from the sidebar first."\n            sources, elapsed = [], 0.0\n            st.markdown(answer)\n        else:\n            placeholder = st.empty()\n            start = time.perf_counter()\n            with st.spinner("Retrieving & generating…"):\n                result = pipeline.query(user_question, top_k=top_k)\n            elapsed = time.perf_counter() - start\n            answer = result["answer"]\n            sources = result["sources"]\n            placeholder.markdown(answer)\n            with st.expander(f" Sources & retrieved chunks · {elapsed:.2f}s"):\n                for src in sources:\n                    st.markdown(\n                        f"<div class=\'source-card\'><b>{src[\'citation\']} {src[\'filename\']}</b> "\n                        f"— chunk #{src[\'chunk_index\']} · score {src[\'score\']}</div>",\n                        unsafe_allow_html=True,\n                    )\n\n    st.session_state.chat_history.append({\n        "role": "assistant", "content": answer, "sources": sources, "elapsed": elapsed,\n    })\n'

with open(r"03_Multi_Document_Research_Agent/streamlit_app.py", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/streamlit_app.py")


## 8. Generate Unit Tests

In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent/tests", exist_ok=True)

_content = ''

with open(r"03_Multi_Document_Research_Agent/tests/__init__.py", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/tests/__init__.py")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent/tests", exist_ok=True)

_content = '"""Unit tests for src/chunker.py"""\n\nimport pytest\n\nfrom src.chunker import Chunker, _split_recursive, _split_semantic\nfrom src.document_loader import Document\n\n\ndef test_recursive_split_respects_chunk_size():\n    text = "word " * 500  # 2500 chars\n    chunks = _split_recursive(text, chunk_size=200, chunk_overlap=20)\n    assert len(chunks) > 1\n    # Allow a little slack since overlap can push chunks slightly over.\n    assert all(len(c) <= 260 for c in chunks)\n\n\ndef test_recursive_split_short_text_returns_single_chunk():\n    text = "short text"\n    chunks = _split_recursive(text, chunk_size=200, chunk_overlap=20)\n    assert chunks == [text]\n\n\ndef test_semantic_split_keeps_sentences_intact():\n    text = "First sentence. Second sentence. Third sentence. Fourth sentence."\n    chunks = _split_semantic(text, chunk_size=30, chunk_overlap=5)\n    for chunk in chunks:\n        assert chunk.strip().endswith((".", "!", "?"))\n\n\ndef test_chunker_invalid_overlap_raises():\n    with pytest.raises(ValueError):\n        Chunker(chunk_size=100, chunk_overlap=100)\n\n\ndef test_chunker_split_document_inherits_metadata():\n    doc = Document(doc_id="doc_abc123", source="/tmp/fake.txt",\n                    text="Sentence one. Sentence two. Sentence three.",\n                    metadata={"filename": "fake.txt"})\n    chunker = Chunker(chunk_size=50, chunk_overlap=5, strategy="semantic")\n    chunks = chunker.split_document(doc)\n    assert len(chunks) >= 1\n    assert all(c.metadata["doc_id"] == "doc_abc123" for c in chunks)\n    assert all(c.metadata["filename"] == "fake.txt" for c in chunks)\n    assert chunks[0].chunk_id == "doc_abc123_chunk_0000"\n'

with open(r"03_Multi_Document_Research_Agent/tests/test_chunker.py", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/tests/test_chunker.py")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent/tests", exist_ok=True)

_content = '"""Unit tests for src/document_loader.py"""\n\nimport pytest\n\nfrom src.document_loader import DocumentLoadError, load_document\nfrom src.utils import clean_text\n\n\ndef test_clean_text_collapses_whitespace():\n    dirty = "Hello   world\\n\\n\\n\\nGoodbye\\x00 world"\n    cleaned = clean_text(dirty)\n    assert "\\x00" not in cleaned\n    assert "\\n\\n\\n" not in cleaned\n    assert "Hello world" in cleaned\n\n\ndef test_load_txt_document(tmp_path):\n    file_path = tmp_path / "sample.txt"\n    file_path.write_text("This is a   test document.\\n\\n\\nWith extra whitespace.")\n\n    document = load_document(file_path)\n\n    assert document.source == str(file_path)\n    assert "test document" in document.text\n    assert document.metadata["file_type"] == "txt"\n    assert document.metadata["num_words"] > 0\n    assert document.doc_id.startswith("doc_")\n\n\ndef test_load_document_missing_file_raises(tmp_path):\n    missing = tmp_path / "does_not_exist.txt"\n    with pytest.raises(DocumentLoadError):\n        load_document(missing)\n\n\ndef test_load_document_unsupported_extension_raises(tmp_path):\n    bad_file = tmp_path / "sample.xyz"\n    bad_file.write_text("data")\n    with pytest.raises(DocumentLoadError):\n        load_document(bad_file)\n'

with open(r"03_Multi_Document_Research_Agent/tests/test_document_loader.py", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/tests/test_document_loader.py")


In [ ]:
import os

os.makedirs(r"03_Multi_Document_Research_Agent/tests", exist_ok=True)

_content = '"""Unit tests for src/retriever.py and src/reranker.py"""\n\nfrom src.reranker import BM25Index, reciprocal_rank_fusion\nfrom src.retriever import compress_context, deduplicate\n\n\ndef _record(chunk_id, text, **extra):\n    return {"chunk_id": chunk_id, "doc_id": "doc_1", "text": text, "metadata": {}, **extra}\n\n\ndef test_deduplicate_removes_near_identical_chunks():\n    records = [\n        _record("c1", "The quick brown fox jumps over the lazy dog."),\n        _record("c2", "The quick brown fox jumps over the lazy dog!"),  # near-duplicate\n        _record("c3", "A completely different sentence about cats."),\n    ]\n    deduped = deduplicate(records)\n    assert len(deduped) == 2\n    assert deduped[0]["chunk_id"] == "c1"\n    assert deduped[1]["chunk_id"] == "c3"\n\n\ndef test_compress_context_respects_char_budget():\n    records = [_record("c1", "x" * 100), _record("c2", "y" * 100), _record("c3", "z" * 100)]\n    compressed = compress_context(records, max_chars=150)\n    total_chars = sum(len(r["text"]) for r in compressed)\n    assert total_chars <= 155  # small slack for ellipsis character\n    assert len(compressed) <= 2\n\n\ndef test_bm25_index_ranks_relevant_document_higher():\n    records = [\n        _record("c1", "Python is a popular programming language for data science."),\n        _record("c2", "Bananas are a good source of potassium."),\n    ]\n    index = BM25Index()\n    index.build(records)\n    results = index.search("programming language", top_k=2)\n    assert results[0]["chunk_id"] == "c1"\n\n\ndef test_reciprocal_rank_fusion_combines_rankings():\n    list_a = [_record("c1", "a"), _record("c2", "b"), _record("c3", "c")]\n    list_b = [_record("c3", "c"), _record("c1", "a"), _record("c2", "b")]\n    fused = reciprocal_rank_fusion([list_a, list_b], weights=[0.5, 0.5])\n    assert {r["chunk_id"] for r in fused} == {"c1", "c2", "c3"}\n    assert fused[0]["fused_score"] >= fused[-1]["fused_score"]\n'

with open(r"03_Multi_Document_Research_Agent/tests/test_retriever.py", "w", encoding="utf-8") as f:
    f.write(_content)

print("Created 03_Multi_Document_Research_Agent/tests/test_retriever.py")


In [ ]:
# Sanity check: confirm every expected file now exists on disk.
expected_files = [
    "README.md", "LICENSE", ".gitignore", "requirements.txt", ".env.example", "setup.py",
    "Dockerfile", "docker-compose.yml", ".github/workflows/python.yml", "configs/config.yaml",
    "src/__init__.py", "src/config.py", "src/logger.py", "src/utils.py",
    "src/document_loader.py", "src/chunker.py", "src/embedder.py", "src/vector_store.py",
    "src/reranker.py", "src/retriever.py", "src/memory.py", "src/prompts.py",
    "src/gemini_client.py", "src/pipeline.py", "src/evaluation.py", "src/visualization.py",
    "main.py", "api.py", "streamlit_app.py",
    "tests/__init__.py", "tests/test_chunker.py", "tests/test_document_loader.py",
    "tests/test_retriever.py",
]
missing = [f for f in expected_files if not (PROJECT_ROOT / f).exists()]
if missing:
    raise RuntimeError(f"Missing generated files: {missing}")
print(f"All {len(expected_files)} project files generated successfully.")

## 9. Make the Project Importable

Add the generated project directory to `sys.path` (so `from src... import ...`
works directly in this notebook) and `cd` into it (so relative paths used by
the CLI/API/Streamlit apps — `data/`, `logs/`, `configs/` — resolve correctly).

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())

## 10. Example Dataset — Sample Documents

To keep this notebook fully self-contained and reproducible (no flaky
external downloads), we generate a small multi-topic example dataset
directly — covering AI/ML, a company refund policy, and renewable energy.
Swap these out for your own PDFs/DOCX/TXT files at any time by uploading
them to `data/documents/` (via the Colab file browser, or the Streamlit
sidebar / FastAPI `/upload` endpoint).

In [ ]:
sample_documents = {
    "ai_and_ml_overview.txt": """
Artificial intelligence (AI) is the simulation of human intelligence processes
by machines, especially computer systems. Machine learning is a subset of AI
that enables systems to learn patterns from data without being explicitly
programmed for every rule. Deep learning uses artificial neural networks with
many layers to model complex, non-linear relationships in large datasets.

Natural language processing (NLP) allows computers to understand, interpret,
and generate human language. Large language models (LLMs) are a class of deep
learning models trained on massive text corpora that can perform a wide range
of language tasks, from summarization to translation to question answering.

Retrieval-augmented generation (RAG) combines a retrieval system — typically a
vector database — with a generative language model. Instead of relying purely
on knowledge memorized during training, a RAG system retrieves relevant
passages from an external document collection and conditions the model's
answer on that retrieved context. This reduces hallucination, improves
factual accuracy, and allows the system to answer questions about documents
the underlying model has never seen before.

Hybrid search combines dense vector similarity search with sparse lexical
search methods like BM25. Vector search excels at capturing semantic meaning
even when exact keywords differ, while BM25 excels at precise keyword and
terminology matching. Fusing both rankings — for example with Reciprocal Rank
Fusion — typically outperforms either method alone.
""",
    "company_refund_policy.txt": """
Our refund policy is designed to be fair and transparent for all customers.

Physical products may be returned within 30 days of delivery for a full
refund, provided the item is unused and in its original packaging. To
initiate a return, contact support with your order number and reason for
the return. Once we receive and inspect the returned item, refunds are
typically processed within 5 to 10 business days and issued to the original
payment method.

Digital products, including software licenses and downloadable content, are
generally non-refundable once the download link has been accessed, except
where required by local consumer protection law.

Subscription services can be cancelled at any time from the account
settings page. Cancelling a subscription stops future billing but does not
automatically refund the current billing period unless the cancellation
occurs within 14 days of the initial purchase (our satisfaction guarantee
window).

Shipping costs are non-refundable unless the return is a result of our
error, such as shipping the wrong item or a defective product. In those
cases, we also cover return shipping costs.
""",
    "renewable_energy_basics.txt": """
Renewable energy comes from naturally replenishing sources such as sunlight,
wind, water, and geothermal heat. Unlike fossil fuels, renewable sources do
not run out on human timescales and typically produce far fewer greenhouse
gas emissions during operation.

Solar panels convert sunlight into electricity using the photovoltaic
effect: photons striking a semiconductor material (usually silicon) knock
electrons loose, generating a flow of electric current. Panel efficiency —
the percentage of sunlight converted into usable electricity — has improved
significantly over the past two decades, with commercial panels now
commonly exceeding 20% efficiency.

Wind turbines convert the kinetic energy of moving air into electricity via
a rotor connected to a generator. Offshore wind farms can capture stronger,
more consistent winds than onshore installations, though they are more
expensive to build and maintain.

Battery storage is a critical complement to renewable generation, since
solar and wind output varies with weather and time of day. Grid-scale
lithium-ion battery installations help smooth out this variability by
storing excess generation for use during low-production periods.
""",
}

documents_dir = PROJECT_ROOT / "data" / "documents"
documents_dir.mkdir(parents=True, exist_ok=True)

for filename, content in sample_documents.items():
    (documents_dir / filename).write_text(content.strip(), encoding="utf-8")

print(f"Wrote {len(sample_documents)} example documents to {documents_dir}")
for f in sorted(documents_dir.glob("*.txt")):
    print("  -", f.name, f"({f.stat().st_size} bytes)")

## 11. Build the Vector Database — Index the Example Documents

This loads every document under `data/documents/`, cleans and chunks the
text, embeds each chunk with `sentence-transformers`, and builds/persists a
FAISS vector index at `data/vectorstore/`. Re-running this cell is safe:
`VectorStore.add()` skips chunks that are already indexed (incremental
indexing).

In [ ]:
from src.pipeline import RAGPipeline

pipeline = RAGPipeline(lazy_gemini=True)   # lazy_gemini=True: don't require a key yet
stats = pipeline.ingest_directory()

print("Ingestion summary:")
for key, value in stats.items():
    print(f"  {key}: {value}")

In [ ]:
# Peek at the retriever with a couple of test queries (retrieval only — no Gemini call yet).
for test_query in ["What is the refund window for returns?", "How does RAG reduce hallucination?"]:
    result = pipeline.retriever.retrieve(test_query, top_k=3)
    print(f"\nQuery: {test_query}")
    print(f"Timing: {result['timing']}")
    for chunk in result["chunks"]:
        preview = chunk["text"][:90].replace(chr(10), " ")
        print(f"  [{chunk['metadata']['filename']}] score={chunk.get('fused_score', 0):.4f} — {preview}...")

## 12. Run the CLI

`main.py` exposes `ingest`, `query`, `chat`, and `evaluate` subcommands. The
cells below run it exactly as a user would from a terminal. **Requires a
valid `GEMINI_API_KEY`** (set in the Configuration section above).

In [ ]:
!python main.py ingest --path data/documents

In [ ]:
import os
from getpass import getpass

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")

print("GEMINI_API_KEY configured")

In [ ]:
!python main.py query "What is the refund window for returns?"

## 13. Run the FastAPI REST Service

Launches `uvicorn` in a background thread inside the notebook process, then
exercises `/health` and `/query` with real HTTP requests. Visit
`http://localhost:8000/docs` for interactive Swagger UI while the server
cell is running (use Colab's port-forwarding / the `ngrok`-based public URL
if you need external access).

In [ ]:
import nest_asyncio
import threading
import uvicorn
import requests

nest_asyncio.apply()

def _run_api_server():
    uvicorn.run("api:app", host="0.0.0.0", port=8000, log_level="warning")

api_thread = threading.Thread(target=_run_api_server, daemon=True)
api_thread.start()
time.sleep(4)  # give the server a moment to boot
print("FastAPI server starting at http://localhost:8000  (docs at /docs)")

In [ ]:
import os
import json
import requests  # re-imported here so this cell is safe to run even after a kernel restart

health = requests.get("http://localhost:8000/health").json()
print("GET /health ->", health)

if os.environ.get("GEMINI_API_KEY", "PASTE_YOUR_API_KEY_HERE") != "PASTE_YOUR_API_KEY_HERE":
    response = requests.post(
        "http://localhost:8000/query",
        json={"question": "How do solar panels generate electricity?", "top_k": 3},
        timeout=60,
    )
    print("\nPOST /query ->")
    print(json.dumps(response.json(), indent=2))
else:
    print("⏭  Skipping /query demo — set GEMINI_API_KEY to test generation.")

In [ ]:
import os
from pathlib import Path

# Get the project root from the existing PROJECT_ROOT variable
project_root_path = Path(os.getcwd())

# Write the GEMINI_API_KEY to a .env file in the project root
env_file_path = project_root_path / ".env"
with open(env_file_path, "w") as f:
    f.write(f"GEMINI_API_KEY={os.environ.get('GEMINI_API_KEY')}\n")
    f.write(f"GEMINI_MODEL={os.environ.get('GEMINI_MODEL')}\n")
    f.write(f"EMBEDDING_MODEL={os.environ.get('EMBEDDING_MODEL')}\n")

print(f"Created .env file at {env_file_path}")

# Stop the existing FastAPI server thread
if 'api_thread' in locals() and api_thread.is_alive():
    # This is a bit hacky, but directly terminating a thread is hard.
    # The uvicorn server should shut down when the thread exits.
    # For robust shutdown in production, consider a more graceful signal handling.
    print("Stopping existing FastAPI server...")
    # As an alternative, if uvicorn stores its server object, we could try server.should_exit = True
    # For this Colab context, relying on the thread's daemon nature and restart seems sufficient.

# Restart the FastAPI server
api_thread = threading.Thread(target=_run_api_server, daemon=True)
api_thread.start()
time.sleep(4)  # give the server a moment to boot
print("FastAPI server restarted with .env configuration.")

In [ ]:
import os
import json
import requests

BASE_URL = "http://localhost:8000"

# ---------------------------------------------------------
# 1. Verify Gemini API key is available (already checked, but for re-run clarity)
# ---------------------------------------------------------
if not os.environ.get("GEMINI_API_KEY"):
    print("GEMINI_API_KEY is not configured.")
    print("Run the secure Configuration cell first.")
else:
    print("GEMINI_API_KEY is available")

    # -----------------------------------------------------
    # 2. Test health endpoint
    # -----------------------------------------------------
    try:
        health_response = requests.get(
            f"{BASE_URL}/health",
            timeout=10
        )

        print("\nGET /health")
        print("Status:", health_response.status_code)

        try:
            print(json.dumps(health_response.json(), indent=2))
        except ValueError:
            print(health_response.text)

    except requests.RequestException as e:
        print(f"Health request failed: {e}")

    # -----------------------------------------------------
    # 3. Test RAG query endpoint
    # -----------------------------------------------------
    try:
        response = requests.post(
            f"{BASE_URL}/query",
            json={
                "question": "How do solar panels generate electricity?",
                "top_k": 3
            },
            timeout=60,
        )

        print("\nPOST /query")
        print("Status:", response.status_code)

        # Safely handle JSON/non-JSON responses
        try:
            data = response.json()
            print(json.dumps(data, indent=2))
        except ValueError:
            print("Server did not return JSON.")
            print("Response:")
            print(response.text)

    except requests.Timeout:
        print(" /query timed out.")

    except requests.RequestException as e:
        print(f" /query request failed: {e}")

## 14. Run the Streamlit Chat UI

Colab can't serve Streamlit's port directly to your browser, so we tunnel it.

**We use [Cloudflare Quick Tunnels](https://developers.cloudflare.com/pages/how-to/preview-with-cloudflare-tunnel/) (`cloudflared`) instead of `localtunnel`.**
`localtunnel`'s free relay is known to drop or time out requests when a page
fires off many small requests at once — which is exactly what Streamlit's UI
does (it lazy-loads each widget — file uploader, chat input, sliders — as its
own JS chunk). That's what caused the repeated
`Failed to fetch dynamically imported module` errors. `cloudflared` handles
concurrent requests reliably, needs no signup/API key, and doesn't show any
interstitial/password page.

Run the two cells below, then open the printed `https://....trycloudflare.com`
URL in a fresh browser tab.


In [ ]:
# One-time setup: download the cloudflared binary (skips if already present).
import os

if not os.path.exists("./cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared
    print("cloudflared downloaded.")
else:
    print("cloudflared already present.")


In [ ]:
# Cleanly (re)start Streamlit + a Cloudflare Quick Tunnel — safe to re-run any time.
import os
import re
import subprocess
import time

os.makedirs("logs", exist_ok=True)

# --- 1. Kill ANY previous Streamlit / tunnel processes first ----------------
!pkill -9 -f "streamlit run" || true
!pkill -9 -f "localtunnel" || true
!pkill -9 -f "cloudflared" || true
time.sleep(2)

# --- 2. Start exactly ONE fresh Streamlit process ---------------------------
streamlit_log = open("logs/streamlit.log", "w")
streamlit_process = subprocess.Popen(
    [
        "streamlit", "run", "streamlit_app.py",
        "--server.port=8501",
        "--server.address=0.0.0.0",
        "--server.headless=true",
    ],
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
)

# Poll instead of a fixed sleep, so we know it's really ready.
ready = False
for _ in range(30):
    result = subprocess.run(
        ["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}", "http://127.0.0.1:8501"],
        capture_output=True, text=True,
    )
    if result.stdout.strip() == "200":
        ready = True
        break
    time.sleep(1)

print("Streamlit is up on port 8501" if ready else " Streamlit did not respond in time — check logs/streamlit.log")

# --- 3. Start exactly ONE Cloudflare Quick Tunnel ----------------------------
get_ipython().system_raw(
    "./cloudflared tunnel --url http://localhost:8501 > logs/cloudflared.log 2>&1 &"
)

tunnel_url = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists("logs/cloudflared.log"):
        log_text = open("logs/cloudflared.log", "r", errors="ignore").read()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_text)
        if match:
            tunnel_url = match.group(0)
            break

if tunnel_url:
    print(f"\nOpen this URL in a NEW / incognito browser tab: {tunnel_url}")
    print("   No password/interstitial needed — it should load directly.")
else:
    print("\nTunnel URL not found yet — re-run this cell, or check logs/cloudflared.log")


## 15. Testing — Unit Test Suite

In [ ]:
!python -m pytest tests/ -v

## 16. Evaluation — Retrieval Quality & Latency

Runs Precision@K / Recall@K / MRR against a small labelled query set, then
benchmarks end-to-end latency and visualizes both the timing breakdown and
the chunk embedding space.

In [ ]:
from src.evaluation import evaluate_retrieval, measure_latency, summarize_latency
from src.visualization import plot_embedding_space, plot_retrieval_timing, plot_precision_recall

# A tiny labelled set for demonstration — in a real project, curate this from
# actual user queries and known-relevant chunk_ids (see vector_store.records).
sample_chunk_ids = [r["chunk_id"] for r in pipeline.vector_store.records if "refund" in r["text"].lower()]

labelled_queries = [
    {"query": "What is the refund window for returns?", "relevant_ids": sample_chunk_ids[:1]}
]

eval_df = evaluate_retrieval(pipeline, labelled_queries, k=3)
eval_df

In [ ]:
fig_pr = plot_precision_recall(eval_df)
fig_pr.show()

In [ ]:
import numpy as np

records = pipeline.vector_store.records
chunk_texts = [r["text"] for r in records]
chunk_labels = [t[:60] + "..." for t in chunk_texts]
chunk_sources = [r["metadata"].get("filename", "unknown") for r in records]

chunk_embeddings = pipeline.embedder.embed_texts(chunk_texts)
fig_embed = plot_embedding_space(chunk_embeddings, chunk_labels, color_by=chunk_sources)
fig_embed.show()

## 17. GitHub Export

Zip the generated project and download it, or push it directly to a new
GitHub repository from within Colab.

In [ ]:
# Stop the background servers before zipping, so log files aren't mid-write.
try:
    streamlit_process.terminate()
except NameError:
    pass

zip_path = shutil.make_archive(str(Path.cwd().parent / PROJECT_NAME), "zip", root_dir=Path.cwd())
print(f"Project zipped: {zip_path}")

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Not running in Colab — find the zip at the path printed above.")